## Connect to the data and analyse fraud by product

In [2]:
import duckdb

# Connect to DuckDB in memory.
# Our source Parquet files will remain unchanged.
con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)
print("Connection established successfully.")

DuckDB version: 1.5.6
Connection established successfully.


## first analytical SQL query

In [3]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS transaction_count,
    SUM(isFraud) AS fraud_count,
    COUNT(*) - SUM(isFraud) AS legitimate_count,
    ROUND(100.0 * AVG(isFraud), 3) AS fraud_rate_pct,
    ROUND(AVG(TransactionAmt), 2) AS average_amount,
    ROUND(MEDIAN(TransactionAmt), 2) AS median_amount
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY ProductCD
ORDER BY transaction_count DESC;
"""

result = con.execute(query).df()

result

,ProductCD,transaction_count,fraud_count,legitimate_count,fraud_rate_pct,average_amount,median_amount
0,W,439670,8969.0,430701.0,2.040,153.16,78.50
1,C,68519,8008.0,60511.0,11.687,42.87,31.19
2,R,37699,1426.0,36273.0,3.783,168.31,125.00
3,H,33024,1574.0,31450.0,4.766,73.17,50.00
4,S,11628,686.0,10942.0,5.900,60.27,35.00


## conditional aggregation

In [4]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS total_transactions,

    SUM(CASE
        WHEN isFraud = 1 THEN 1
        ELSE 0
    END) AS fraud_transactions,

    SUM(CASE
        WHEN isFraud = 0 THEN 1
        ELSE 0
    END) AS legitimate_transactions,

    ROUND(
        100.0 * SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        3
    ) AS fraud_rate_pct,

    ROUND(
        100.0 * SUM(CASE WHEN TransactionAmt >= 100 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS transactions_100_or_more_pct

FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY ProductCD
ORDER BY fraud_rate_pct DESC;
"""

con.execute(query).df()

,ProductCD,total_transactions,fraud_transactions,legitimate_transactions,fraud_rate_pct,transactions_100_or_more_pct
0,C,68519,8008.0,60511.0,11.687,7.25
1,S,11628,686.0,10942.0,5.900,24.10
2,H,33024,1574.0,31450.0,4.766,30.28
3,R,37699,1426.0,36273.0,3.783,84.42
4,W,439670,8969.0,430701.0,2.040,43.83


## WHERE, DISTINCT, and HAVING

In [5]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS transaction_count,
    SUM(isFraud) AS fraud_count,
    ROUND(AVG(TransactionAmt), 2) AS average_amount
FROM read_parquet('data/interim/train_transaction.parquet')
WHERE TransactionAmt >= 100
GROUP BY ProductCD
HAVING COUNT(*) >= 1000
ORDER BY fraud_count DESC;
"""

result = con.execute(query).df()
result

,ProductCD,transaction_count,fraud_count,average_amount
0,W,192688,5155.0,282.94
1,R,31825,1353.0,189.18
2,H,9999,1097.0,141.41
3,C,4969,812.0,148.53
4,S,2802,148.0,151.43


### Business question: Among transactions worth at least 100, which product categories have the highest fraud rate, and how many fraudulent transactions occur in each category?

In [6]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS qualifying_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct,
    ROUND(
        SUM(CASE
            WHEN isFraud = 1 THEN TransactionAmt
            ELSE 0
        END),
        2
    ) AS total_fraud_amount
FROM read_parquet('data/interim/train_transaction.parquet')
WHERE TransactionAmt >= 100
GROUP BY ProductCD
HAVING COUNT(*) >= 1000
ORDER BY fraud_rate_pct DESC;
"""

result = con.execute(query).df()
result

,ProductCD,qualifying_transactions,fraud_count,fraud_rate_pct,total_fraud_amount
0,C,4969,812.0,16.341,123485.89
1,H,9999,1097.0,10.971,222966.00
2,S,2802,148.0,5.282,25230.00
3,R,31825,1353.0,4.251,344300.00
4,W,192688,5155.0,2.675,1836474.60


## Among transactions of 100 or more, which product category has the highest fraud rate?

In [7]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS qualifying_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet')
WHERE TransactionAmt >= 100
GROUP BY ProductCD
HAVING COUNT(*) >= 1000
ORDER BY fraud_rate_pct DESC
LIMIT 1;
"""

con.execute(query).df()

,ProductCD,qualifying_transactions,fraud_count,fraud_rate_pct
0,C,4969,812.0,16.341


## Which card type has the highest proportion of fraudulent transactions, and how many fraud cases does it account for?

In [10]:
query = """
SELECT
    COALESCE(card4, 'MISSING') AS card_type,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY COALESCE(card4, 'MISSING')
ORDER BY fraud_rate_pct DESC
LIMIT 1;
"""

result = con.execute(query).df()
result

,card_type,total_transactions,fraud_count,fraud_rate_pct
0,discover,6651,514.0,7.728


## How do fraud rates and fraud counts compare across all card types, including missing card-type values?

In [11]:
query = """
SELECT
    COALESCE(card4, 'MISSING') AS card_type,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    COUNT(*) - SUM(isFraud) AS legitimate_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY COALESCE(card4, 'MISSING')
ORDER BY fraud_rate_pct DESC;
"""

result = con.execute(query).df()
result

,card_type,total_transactions,fraud_count,legitimate_count,fraud_rate_pct
0,discover,6651,514.0,6137.0,7.728
1,visa,384767,13373.0,371394.0,3.476
2,mastercard,189217,6496.0,182721.0,3.433
3,american express,8328,239.0,8089.0,2.870
4,MISSING,1577,41.0,1536.0,2.600


## How does the fraud rate change across transaction-amount bands, and which bands contribute the most fraud cases?

In [12]:
query = """
WITH transaction_bands AS (
    SELECT
        CASE
            WHEN TransactionAmt < 25 THEN '01: Under 25'
            WHEN TransactionAmt < 100 THEN '02: 25 to under 100'
            WHEN TransactionAmt < 250 THEN '03: 100 to under 250'
            WHEN TransactionAmt < 500 THEN '04: 250 to under 500'
            ELSE '05: 500 and above'
        END AS amount_band,
        isFraud
    FROM read_parquet('data/interim/train_transaction.parquet')
)
SELECT
    amount_band,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM transaction_bands
GROUP BY amount_band
ORDER BY amount_band;
"""

result = con.execute(query).df()
result

,amount_band,total_transactions,fraud_count,fraud_rate_pct
0,01: Under 25,43329,3019.0,6.968
1,02: 25 to under 100,304928,9079.0,2.977
2,03: 100 to under 250,175923,5198.0,2.955
3,04: 250 to under 500,42036,2218.0,5.276
4,05: 500 and above,24324,1149.0,4.724


## Does the high fraud rate for transactions under 25 occur across product categories, or is it concentrated in one category?

In [13]:
query = """
SELECT
    ProductCD,
    CASE
        WHEN TransactionAmt < 25 THEN '01: Under 25'
        WHEN TransactionAmt < 100 THEN '02: 25 to under 100'
        WHEN TransactionAmt < 250 THEN '03: 100 to under 250'
        WHEN TransactionAmt < 500 THEN '04: 250 to under 500'
        ELSE '05: 500 and above'
    END AS amount_band,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY
    ProductCD,
    CASE
        WHEN TransactionAmt < 25 THEN '01: Under 25'
        WHEN TransactionAmt < 100 THEN '02: 25 to under 100'
        WHEN TransactionAmt < 250 THEN '03: 100 to under 250'
        WHEN TransactionAmt < 500 THEN '04: 250 to under 500'
        ELSE '05: 500 and above'
    END
ORDER BY ProductCD, amount_band;
"""

result = con.execute(query).df()
result

,ProductCD,amount_band,total_transactions,fraud_count,fraud_rate_pct
0,C,01: Under 25,26037,2621.0,10.066
1,C,02: 25 to under 100,37513,4575.0,12.196
2,C,03: 100 to under 250,4736,755.0,15.942
3,C,04: 250 to under 500,231,55.0,23.810
4,C,05: 500 and above,2,2.0,100.000
5,H,01: Under 25,1615,43.0,2.663
6,H,02: 25 to under 100,21410,434.0,2.027
7,H,03: 100 to under 250,9070,760.0,8.379
8,H,04: 250 to under 500,880,323.0,36.705
9,H,05: 500 and above,49,14.0,28.571


## Which product-and-amount groups have elevated fraud rates while also containing enough transactions to justify further investigation?

In [14]:
query = """
WITH product_amount_analysis AS (
    SELECT
        ProductCD,
        CASE
            WHEN TransactionAmt < 25 THEN '01: Under 25'
            WHEN TransactionAmt < 100 THEN '02: 25 to under 100'
            WHEN TransactionAmt < 250 THEN '03: 100 to under 250'
            WHEN TransactionAmt < 500 THEN '04: 250 to under 500'
            ELSE '05: 500 and above'
        END AS amount_band,
        COUNT(*) AS total_transactions,
        SUM(isFraud) AS fraud_count
    FROM read_parquet('data/interim/train_transaction.parquet')
    GROUP BY
        ProductCD,
        CASE
            WHEN TransactionAmt < 25 THEN '01: Under 25'
            WHEN TransactionAmt < 100 THEN '02: 25 to under 100'
            WHEN TransactionAmt < 250 THEN '03: 100 to under 250'
            WHEN TransactionAmt < 500 THEN '04: 250 to under 500'
            ELSE '05: 500 and above'
        END
)
SELECT
    ProductCD,
    amount_band,
    total_transactions,
    fraud_count,
    ROUND(
        100.0 * fraud_count / total_transactions,
        3
    ) AS fraud_rate_pct
FROM product_amount_analysis
WHERE total_transactions >= 500
ORDER BY fraud_rate_pct DESC;
"""

result = con.execute(query).df()
result

,ProductCD,amount_band,total_transactions,fraud_count,fraud_rate_pct
0,H,04: 250 to under 500,880,323.0,36.705
1,C,03: 100 to under 250,4736,755.0,15.942
2,C,02: 25 to under 100,37513,4575.0,12.196
3,C,01: Under 25,26037,2621.0,10.066
4,R,04: 250 to under 500,5136,500.0,9.735
5,H,03: 100 to under 250,9070,760.0,8.379
6,R,05: 500 and above,1652,110.0,6.659
7,S,01: Under 25,3288,204.0,6.204
8,S,02: 25 to under 100,5538,334.0,6.031
9,S,03: 100 to under 250,2496,128.0,5.128


## How do transaction volume and fraud rate change over successive periods in the training data?

In [15]:
query = """
WITH transactions_by_time AS (
    SELECT
        TransactionDT,
        isFraud,
        NTILE(10) OVER (
            ORDER BY TransactionDT, TransactionID
        ) AS time_group
    FROM read_parquet('data/interim/train_transaction.parquet')
)
SELECT
    time_group,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct,
    MIN(TransactionDT) AS period_start_dt,
    MAX(TransactionDT) AS period_end_dt
FROM transactions_by_time
GROUP BY time_group
ORDER BY time_group;
"""

result = con.execute(query).df()
result

,time_group,total_transactions,fraud_count,fraud_rate_pct,period_start_dt,period_end_dt
0,1,59054,1631.0,2.762,86400,1360999
1,2,59054,1195.0,2.024,1361005,2310138
2,3,59054,2200.0,3.725,2310165,3864159
3,4,59054,2532.0,4.288,3864166,5592303
4,5,59054,2336.0,3.956,5592304,7306520
5,6,59054,2094.0,3.546,7306535,8745772
6,7,59054,2550.0,4.318,8745798,10437996
7,8,59054,2061.0,3.490,10438003,12192842
8,9,59054,1851.0,3.134,12192900,13990904
9,10,59054,2213.0,3.747,13990941,15811131


## Is the change in fraud risk happening within products, or because the mix of products changes over time?

In [16]:
query = """
WITH transactions_by_time AS (
    SELECT
        TransactionID,
        TransactionDT,
        ProductCD,
        isFraud,
        NTILE(10) OVER (
            ORDER BY TransactionDT, TransactionID
        ) AS time_group
    FROM read_parquet('data/interim/train_transaction.parquet')
)
SELECT
    time_group,
    ProductCD,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY time_group),
        2
    ) AS product_share_pct
FROM transactions_by_time
GROUP BY
    time_group,
    ProductCD
ORDER BY
    time_group,
    ProductCD;
"""

result = con.execute(query).df()
result

,time_group,ProductCD,total_transactions,fraud_count,fraud_rate_pct,product_share_pct
0,1,C,5919,628.0,10.610,10.02
1,1,H,9015,152.0,1.686,15.27
2,1,R,4612,95.0,2.060,7.81
3,1,S,1699,29.0,1.707,2.88
4,1,W,37809,727.0,1.923,64.02
5,2,C,7450,473.0,6.349,12.62
6,2,H,10765,178.0,1.654,18.23
7,2,R,14218,71.0,0.499,24.08
8,2,S,1111,36.0,3.240,1.88
9,2,W,25510,437.0,1.713,43.20


## Does missing data itself carry fraud information?

In [17]:
query = """
WITH feature_missingness AS (
    SELECT
        isFraud,
        'card4' AS feature,
        card4 IS NULL AS is_missing
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'card6', card6 IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'P_emaildomain', P_emaildomain IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'R_emaildomain', R_emaildomain IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'M4', M4 IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'D7', D7 IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'dist2', dist2 IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT isFraud, 'V1', V1 IS NULL
    FROM read_parquet('data/interim/train_transaction.parquet')
),
missingness_summary AS (
    SELECT
        feature,
        isFraud,
        COUNT(*) AS transaction_count,
        SUM(CASE WHEN is_missing THEN 1 ELSE 0 END) AS missing_count,
        ROUND(
            100.0 * SUM(CASE WHEN is_missing THEN 1 ELSE 0 END)
            / COUNT(*),
            3
        ) AS missing_rate_pct
    FROM feature_missingness
    GROUP BY feature, isFraud
)
SELECT
    feature,
    MAX(CASE WHEN isFraud = 0
        THEN transaction_count END) AS legitimate_count,
    MAX(CASE WHEN isFraud = 0
        THEN missing_rate_pct END) AS legitimate_missing_pct,
    MAX(CASE WHEN isFraud = 1
        THEN transaction_count END) AS fraud_count,
    MAX(CASE WHEN isFraud = 1
        THEN missing_rate_pct END) AS fraud_missing_pct,
    ROUND(
        MAX(CASE WHEN isFraud = 1
            THEN missing_rate_pct END)
        -
        MAX(CASE WHEN isFraud = 0
            THEN missing_rate_pct END),
        3
    ) AS missing_rate_difference_pp
FROM missingness_summary
GROUP BY feature
ORDER BY ABS(
    MAX(CASE WHEN isFraud = 1
        THEN missing_rate_pct END)
    -
    MAX(CASE WHEN isFraud = 0
        THEN missing_rate_pct END)
) DESC;
"""

result = con.execute(query).df()
result

,feature,legitimate_count,legitimate_missing_pct,fraud_count,fraud_missing_pct,missing_rate_difference_pp
0,R_emaildomain,569877,77.879,20663,45.666,-32.213
1,V1,569877,46.454,20663,70.450,23.996
2,M4,569877,48.470,20663,25.296,-23.174
3,D7,569877,94.187,20663,71.979,-22.208
4,dist2,569877,94.052,20663,81.944,-12.108
5,P_emaildomain,569877,16.085,20663,13.502,-2.583
6,card6,569877,0.269,20663,0.189,-0.080
7,card4,569877,0.270,20663,0.198,-0.072


## How much does fraud risk differ when a feature is missing versus present? “Among transactions where the feature is missing, what percentage are fraudulent?”

In [18]:
query = """
WITH feature_status AS (
    SELECT
        isFraud,
        'V1' AS feature,
        CASE WHEN V1 IS NULL THEN 'Missing' ELSE 'Present' END AS status
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT
        isFraud,
        'M4',
        CASE WHEN M4 IS NULL THEN 'Missing' ELSE 'Present' END
    FROM read_parquet('data/interim/train_transaction.parquet')

    UNION ALL

    SELECT
        isFraud,
        'R_emaildomain',
        CASE
            WHEN R_emaildomain IS NULL THEN 'Missing'
            ELSE 'Present'
        END
    FROM read_parquet('data/interim/train_transaction.parquet')
),
risk_by_status AS (
    SELECT
        feature,
        status,
        COUNT(*) AS total_transactions,
        SUM(isFraud) AS fraud_count,
        ROUND(
            100.0 * SUM(isFraud) / COUNT(*),
            3
        ) AS fraud_rate_pct
    FROM feature_status
    GROUP BY feature, status
)
SELECT
    feature,
    status,
    total_transactions,
    fraud_count,
    fraud_rate_pct
FROM risk_by_status
ORDER BY
    feature,
    status;
"""

result = con.execute(query).df()
result

,feature,status,total_transactions,fraud_count,fraud_rate_pct
0,M4,Missing,281444,5227.0,1.857
1,M4,Present,309096,15436.0,4.994
2,R_emaildomain,Missing,453249,9436.0,2.082
3,R_emaildomain,Present,137291,11227.0,8.178
4,V1,Missing,279287,14557.0,5.212
5,V1,Present,311253,6106.0,1.962


## Does the V1 missingness signal remain useful over time?

In [19]:
query = """
WITH transactions_by_time AS (
    SELECT
        TransactionID,
        TransactionDT,
        isFraud,
        V1,
        NTILE(10) OVER (
            ORDER BY TransactionDT, TransactionID
        ) AS time_group
    FROM read_parquet('data/interim/train_transaction.parquet')
)
SELECT
    time_group,
    CASE
        WHEN V1 IS NULL THEN 'Missing'
        ELSE 'Present'
    END AS v1_status,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM transactions_by_time
GROUP BY
    time_group,
    CASE
        WHEN V1 IS NULL THEN 'Missing'
        ELSE 'Present'
    END
ORDER BY
    time_group,
    v1_status;
"""

result = con.execute(query).df()
result

,time_group,v1_status,total_transactions,fraud_count,fraud_rate_pct
0,1,Missing,40923,1304.0,3.186
1,1,Present,18131,327.0,1.804
2,2,Missing,44658,980.0,2.194
3,2,Present,14396,215.0,1.493
4,3,Missing,33585,1773.0,5.279
5,3,Present,25469,427.0,1.677
6,4,Missing,35882,2015.0,5.616
7,4,Present,23172,517.0,2.231
8,5,Missing,30912,1775.0,5.742
9,5,Present,28142,561.0,1.993


## Is the V1 missingness signal still present within each product?

In [20]:
query = """
SELECT
    ProductCD,
    CASE
        WHEN V1 IS NULL THEN 'Missing'
        ELSE 'Present'
    END AS v1_status,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY
    ProductCD,
    CASE
        WHEN V1 IS NULL THEN 'Missing'
        ELSE 'Present'
    END
ORDER BY
    ProductCD,
    v1_status;
"""

result = con.execute(query).df()
result

,ProductCD,v1_status,total_transactions,fraud_count,fraud_rate_pct
0,C,Missing,68519,8008.0,11.687
1,H,Missing,33024,1574.0,4.766
2,R,Missing,37699,1426.0,3.783
3,S,Missing,11628,686.0,5.900
4,W,Missing,128417,2863.0,2.229
5,W,Present,311253,6106.0,1.962


## verify missingness by product

In [21]:
query = """
SELECT
    ProductCD,
    COUNT(*) AS total_transactions,
    COUNT(V1) AS v1_present_count,
    COUNT(*) - COUNT(V1) AS v1_missing_count,
    ROUND(
        100.0 * (COUNT(*) - COUNT(V1)) / COUNT(*),
        3
    ) AS v1_missing_pct
FROM read_parquet('data/interim/train_transaction.parquet')
GROUP BY ProductCD
ORDER BY ProductCD;
"""

result = con.execute(query).df()
result

,ProductCD,total_transactions,v1_present_count,v1_missing_count,v1_missing_pct
0,C,68519,0,68519,100.000
1,H,33024,0,33024,100.000
2,R,37699,0,37699,100.000
3,S,11628,0,11628,100.000
4,W,439670,311253,128417,29.208


## Does identity-data availability relate to fraud

In [22]:
query = """
WITH transaction_identity AS (
    SELECT
        t.TransactionID,
        t.isFraud,
        CASE
            WHEN i.TransactionID IS NOT NULL THEN 'Identity available'
            ELSE 'Identity unavailable'
        END AS identity_status
    FROM read_parquet('data/interim/train_transaction.parquet') AS t
    LEFT JOIN read_parquet('data/interim/train_identity.parquet') AS i
        ON t.TransactionID = i.TransactionID
)
SELECT
    identity_status,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    ROUND(
        100.0 * SUM(isFraud) / COUNT(*),
        3
    ) AS fraud_rate_pct,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS share_of_all_transactions_pct
FROM transaction_identity
GROUP BY identity_status
ORDER BY identity_status;
"""

result = con.execute(query).df()
result

,identity_status,total_transactions,fraud_count,fraud_rate_pct,share_of_all_transactions_pct
0,Identity available,144233,11318.0,7.847,24.42
1,Identity unavailable,446307,9345.0,2.094,75.58


## Is the identity table safe to join without duplicating transactions?

In [23]:
query = """
SELECT
    COUNT(*) AS identity_rows,
    COUNT(TransactionID) AS non_null_transaction_ids,
    COUNT(DISTINCT TransactionID) AS unique_transaction_ids,
    COUNT(*) - COUNT(DISTINCT TransactionID) AS duplicate_key_excess,
    SUM(
        CASE WHEN TransactionID IS NULL THEN 1 ELSE 0 END
    ) AS null_transaction_ids
FROM read_parquet('data/interim/train_identity.parquet');
"""

result = con.execute(query).df()
result

,identity_rows,non_null_transaction_ids,unique_transaction_ids,duplicate_key_excess,null_transaction_ids
0,144233,144233,144233,0,0.0


## Verify that the join preserves all training transactions

In [24]:
query = """
SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT t.TransactionID) AS unique_transaction_ids,
    SUM(
        CASE
            WHEN i.TransactionID IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS matched_identity_rows,
    SUM(
        CASE
            WHEN i.TransactionID IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_identity_rows,
    COUNT(*) - COUNT(DISTINCT t.TransactionID)
        AS excess_rows
FROM read_parquet('data/interim/train_transaction.parquet') AS t
LEFT JOIN read_parquet('data/interim/train_identity.parquet') AS i
    ON t.TransactionID = i.TransactionID;
"""

result = con.execute(query).df()
result

,joined_rows,unique_transaction_ids,matched_identity_rows,unmatched_identity_rows,excess_rows
0,590540,590540,144233.0,446307.0,0


## Understand which identity features are available

In [25]:
query = """
DESCRIBE
SELECT *
FROM read_parquet('data/interim/train_identity.parquet');
"""

result = con.execute(query).df()
result

,column_name,column_type,null,key,default,extra
0,TransactionID,BIGINT,YES,None,None,None
1,id_01,DOUBLE,YES,None,None,None
2,id_02,DOUBLE,YES,None,None,None
3,id_03,DOUBLE,YES,None,None,None
4,id_04,DOUBLE,YES,None,None,None
5,id_05,DOUBLE,YES,None,None,None
6,id_06,DOUBLE,YES,None,None,None
7,id_07,DOUBLE,YES,None,None,None
8,id_08,DOUBLE,YES,None,None,None
9,id_09,DOUBLE,YES,None,None,None


## Which identity features have the most missing values?

In [26]:
query = """
SELECT
    feature,
    COUNT(*) AS total_transactions,
    COUNT(value) AS present_count,
    COUNT(*) - COUNT(value) AS missing_count,
    ROUND(
        100.0 * (COUNT(*) - COUNT(value)) / COUNT(*),
        2
    ) AS missing_pct
FROM (
    UNPIVOT (
        SELECT
            * EXCLUDE (TransactionID)
            REPLACE (
                CAST(id_01 AS VARCHAR) AS id_01,
                CAST(id_02 AS VARCHAR) AS id_02,
                CAST(id_03 AS VARCHAR) AS id_03,
                CAST(id_04 AS VARCHAR) AS id_04,
                CAST(id_05 AS VARCHAR) AS id_05,
                CAST(id_06 AS VARCHAR) AS id_06,
                CAST(id_07 AS VARCHAR) AS id_07,
                CAST(id_08 AS VARCHAR) AS id_08,
                CAST(id_09 AS VARCHAR) AS id_09,
                CAST(id_10 AS VARCHAR) AS id_10,
                CAST(id_11 AS VARCHAR) AS id_11,
                CAST(id_13 AS VARCHAR) AS id_13,
                CAST(id_14 AS VARCHAR) AS id_14,
                CAST(id_17 AS VARCHAR) AS id_17,
                CAST(id_18 AS VARCHAR) AS id_18,
                CAST(id_19 AS VARCHAR) AS id_19,
                CAST(id_20 AS VARCHAR) AS id_20,
                CAST(id_21 AS VARCHAR) AS id_21,
                CAST(id_22 AS VARCHAR) AS id_22,
                CAST(id_24 AS VARCHAR) AS id_24,
                CAST(id_25 AS VARCHAR) AS id_25,
                CAST(id_26 AS VARCHAR) AS id_26,
                CAST(id_32 AS VARCHAR) AS id_32
            )
            FROM read_parquet('data/interim/train_identity.parquet')
    )
    ON COLUMNS(*)
    INTO NAME feature VALUE value
)
GROUP BY feature
ORDER BY missing_pct DESC, feature;
"""

result = con.execute(query).df()
result

,feature,total_transactions,present_count,missing_count,missing_pct
0,DeviceInfo,118666,118666,0,0.0
1,DeviceType,140810,140810,0,0.0
2,id_01,144233,144233,0,0.0
3,id_02,140872,140872,0,0.0
4,id_03,66324,66324,0,0.0
5,id_04,66324,66324,0,0.0
6,id_05,136865,136865,0,0.0
7,id_06,136865,136865,0,0.0
8,id_07,5155,5155,0,0.0
9,id_08,5155,5155,0,0.0


## verify missingness for one field

In [27]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DeviceInfo) AS non_null_device_info,
    COUNT(*) - COUNT(DeviceInfo) AS null_device_info,
    ROUND(
        100.0 * (COUNT(*) - COUNT(DeviceInfo)) / COUNT(*),
        2
    ) AS missing_pct
FROM read_parquet('data/interim/train_identity.parquet');
"""

result = con.execute(query).df()
result

,total_rows,non_null_device_info,null_device_info,missing_pct
0,144233,118666,25567,17.73


## Can we correctly measure missingness across all identity features?

In [28]:
query = """
SELECT
    'id_01' AS feature,
    COUNT(*) AS total_rows,
    COUNT(id_01) AS present_count,
    COUNT(*) - COUNT(id_01) AS missing_count,
    ROUND(100.0 * (COUNT(*) - COUNT(id_01)) / COUNT(*), 2) AS missing_pct
FROM read_parquet('data/interim/train_identity.parquet')

UNION ALL

SELECT
    'id_12',
    COUNT(*),
    COUNT(id_12),
    COUNT(*) - COUNT(id_12),
    ROUND(100.0 * (COUNT(*) - COUNT(id_12)) / COUNT(*), 2)
FROM read_parquet('data/interim/train_identity.parquet')

UNION ALL

SELECT
    'DeviceInfo',
    COUNT(*),
    COUNT(DeviceInfo),
    COUNT(*) - COUNT(DeviceInfo),
    ROUND(100.0 * (COUNT(*) - COUNT(DeviceInfo)) / COUNT(*), 2)
FROM read_parquet('data/interim/train_identity.parquet')

ORDER BY missing_pct DESC;
"""

result = con.execute(query).df()
result

,feature,total_rows,present_count,missing_count,missing_pct
0,DeviceInfo,144233,118666,25567,17.73
1,id_12,144233,144233,0,0.00
2,id_01,144233,144233,0,0.00


## Profile missingness across all identity features

In [29]:
# Read the actual feature names from the Parquet schema.
schema = con.execute("""
    DESCRIBE
    SELECT * EXCLUDE (TransactionID)
    FROM read_parquet('data/interim/train_identity.parquet')
""").df()

features = schema["column_name"].tolist()

# Build one direct SQL count per feature.
parts = []

for feature in features:
    parts.append(f"""
        SELECT
            '{feature}' AS feature,
            COUNT(*) AS total_rows,
            COUNT("{feature}") AS present_count,
            COUNT(*) - COUNT("{feature}") AS missing_count,
            ROUND(
                100.0 * (COUNT(*) - COUNT("{feature}")) / COUNT(*),
                2
            ) AS missing_pct
        FROM read_parquet('data/interim/train_identity.parquet')
    """)

query = "\nUNION ALL\n".join(parts) + """
ORDER BY missing_pct DESC, feature;
"""

result = con.execute(query).df()
result

,feature,total_rows,present_count,missing_count,missing_pct
0,id_24,144233,4747,139486,96.71
1,id_25,144233,5132,139101,96.44
2,id_07,144233,5155,139078,96.43
3,id_08,144233,5155,139078,96.43
4,id_21,144233,5159,139074,96.42
5,id_22,144233,5169,139064,96.42
6,id_23,144233,5169,139064,96.42
7,id_26,144233,5163,139070,96.42
8,id_27,144233,5169,139064,96.42
9,id_18,144233,45113,99120,68.72


## Does identity-feature missingness differ between fraudulent and legitimate transactions?

In [30]:
schema = con.execute("""
    DESCRIBE
    SELECT * EXCLUDE (TransactionID)
    FROM read_parquet('data/interim/train_identity.parquet')
""").df()

features = schema["column_name"].tolist()

parts = []

for feature in features:
    parts.append(f"""
        SELECT
            '{feature}' AS feature,
            SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END)
                AS fraud_rows,
            SUM(CASE WHEN t.isFraud = 0 THEN 1 ELSE 0 END)
                AS legitimate_rows,
            ROUND(
                100.0 * SUM(
                    CASE WHEN t.isFraud = 1
                              AND i."{feature}" IS NULL
                         THEN 1 ELSE 0 END
                ) / NULLIF(
                    SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END), 0
                ),
                2
            ) AS fraud_missing_pct,
            ROUND(
                100.0 * SUM(
                    CASE WHEN t.isFraud = 0
                              AND i."{feature}" IS NULL
                         THEN 1 ELSE 0 END
                ) / NULLIF(
                    SUM(CASE WHEN t.isFraud = 0 THEN 1 ELSE 0 END), 0
                ),
                2
            ) AS legitimate_missing_pct
        FROM read_parquet('data/interim/train_transaction.parquet') t
        INNER JOIN read_parquet('data/interim/train_identity.parquet') i
            ON t.TransactionID = i.TransactionID
    """)

query = """
SELECT *,
       ROUND(fraud_missing_pct - legitimate_missing_pct, 2)
           AS missing_rate_difference_pp
FROM (
""" + "\nUNION ALL\n".join(parts) + """
) AS feature_missingness
ORDER BY ABS(fraud_missing_pct - legitimate_missing_pct) DESC;
"""

result = con.execute(query).df()
result

,feature,fraud_rows,legitimate_rows,fraud_missing_pct,legitimate_missing_pct,missing_rate_difference_pp
0,id_14,11318.0,132915.0,68.38,42.47,25.91
1,id_30,11318.0,132915.0,69.59,44.23,25.36
2,id_32,11318.0,132915.0,69.51,44.22,25.29
3,id_34,11318.0,132915.0,69.21,44.08,25.13
4,id_33,11318.0,132915.0,70.29,47.39,22.90
5,id_10,11318.0,132915.0,30.84,49.52,-18.68
6,id_09,11318.0,132915.0,30.84,49.52,-18.68
7,id_04,11318.0,132915.0,37.16,55.45,-18.29
8,id_03,11318.0,132915.0,37.16,55.45,-18.29
9,DeviceInfo,11318.0,132915.0,23.95,17.20,6.75


## Does DeviceType have different fraud rates across its categories?

In [31]:
query = """
SELECT
    COALESCE(i.DeviceType, 'MISSING') AS device_type,
    COUNT(*) AS transaction_count,
    SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(
        100.0 * SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet') t
INNER JOIN read_parquet('data/interim/train_identity.parquet') i
    ON t.TransactionID = i.TransactionID
GROUP BY COALESCE(i.DeviceType, 'MISSING')
ORDER BY transaction_count DESC;
"""

result = con.execute(query).df()
result

,device_type,transaction_count,fraud_count,fraud_rate_pct
0,desktop,85165,5554.0,6.52
1,mobile,55645,5657.0,10.17
2,MISSING,3423,107.0,3.13


## Is the device-type pattern stable over time?

In [32]:
query = """
WITH identity_transactions AS (
    SELECT
        t.TransactionID,
        t.TransactionDT,
        t.isFraud,
        COALESCE(i.DeviceType, 'MISSING') AS device_type,
        NTILE(10) OVER (
            ORDER BY t.TransactionDT, t.TransactionID
        ) AS time_group
    FROM read_parquet('data/interim/train_transaction.parquet') t
    INNER JOIN read_parquet('data/interim/train_identity.parquet') i
        ON t.TransactionID = i.TransactionID
)
SELECT
    time_group,
    device_type,
    COUNT(*) AS transaction_count,
    SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(
        100.0 * SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM identity_transactions
GROUP BY time_group, device_type
ORDER BY time_group, device_type;
"""

result = con.execute(query).df()
result

,time_group,device_type,transaction_count,fraud_count,fraud_rate_pct
0,1,MISSING,343,3.0,0.87
1,1,desktop,9403,319.0,3.39
2,1,mobile,4678,303.0,6.48
3,2,MISSING,351,2.0,0.57
4,2,desktop,9567,267.0,2.79
5,2,mobile,4506,218.0,4.84
6,3,MISSING,507,4.0,0.79
7,3,desktop,9789,184.0,1.88
8,3,mobile,4128,138.0,3.34
9,4,MISSING,276,2.0,0.72


## Does DeviceInfo add useful categorical information?

In [33]:
query = """
SELECT
    COALESCE(DeviceInfo, 'MISSING') AS device_info,
    COUNT(*) AS transaction_count,
    SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(
        100.0 * SUM(CASE WHEN t.isFraud = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM read_parquet('data/interim/train_transaction.parquet') t
INNER JOIN read_parquet('data/interim/train_identity.parquet') i
    ON t.TransactionID = i.TransactionID
GROUP BY COALESCE(DeviceInfo, 'MISSING')
ORDER BY transaction_count DESC
LIMIT 20;
"""

result = con.execute(query).df()
result

,device_info,transaction_count,fraud_count,fraud_rate_pct
0,Windows,47722,3121.0,6.54
1,MISSING,25567,2711.0,10.60
2,iOS Device,19782,1240.0,6.27
3,MacOS,12573,278.0,2.21
4,Trident/7.0,7440,96.0,1.29
5,rv:11.0,1901,76.0,4.00
6,rv:57.0,962,103.0,10.71
7,SM-J700M Build/MMB29K,549,60.0,10.93
8,SM-G610M Build/MMB29K,461,37.0,8.03
9,SM-G531H Build/LMY48B,410,33.0,8.05


## How many distinct device-information values are there?

In [34]:
query = """
WITH device_counts AS (
    SELECT
        DeviceInfo,
        COUNT(*) AS transaction_count
    FROM read_parquet('data/interim/train_identity.parquet')
    WHERE DeviceInfo IS NOT NULL
    GROUP BY DeviceInfo
)
SELECT
    COUNT(*) AS distinct_device_info_values,
    SUM(transaction_count) AS transactions_with_device_info,
    SUM(CASE WHEN transaction_count = 1 THEN 1 ELSE 0 END)
        AS values_seen_once,
    SUM(CASE WHEN transaction_count < 10 THEN 1 ELSE 0 END)
        AS values_seen_fewer_than_10_times,
    ROUND(
        100.0 * SUM(
            CASE WHEN transaction_count < 10 THEN transaction_count ELSE 0 END
        ) / SUM(transaction_count),
        2
    ) AS pct_transactions_in_rare_values
FROM device_counts;
"""

result = con.execute(query).df()
result

,distinct_device_info_values,transactions_with_device_info,values_seen_once,values_seen_fewer_than_10_times,pct_transactions_in_rare_values
0,1786,118666.0,440.0,1273.0,3.52


## How much of the device information is concentrated in the most common categories?

In [35]:
query = """
WITH device_counts AS (
    SELECT
        DeviceInfo,
        COUNT(*) AS transaction_count
    FROM read_parquet('data/interim/train_identity.parquet')
    WHERE DeviceInfo IS NOT NULL
    GROUP BY DeviceInfo
),
ranked_devices AS (
    SELECT
        DeviceInfo,
        transaction_count,
        ROW_NUMBER() OVER (
            ORDER BY transaction_count DESC, DeviceInfo
        ) AS device_rank
    FROM device_counts
)
SELECT
    COUNT(*) AS distinct_values,
    SUM(transaction_count) AS total_transactions,
    SUM(CASE WHEN device_rank <= 10 THEN transaction_count ELSE 0 END)
        AS top_10_transactions,
    ROUND(
        100.0 * SUM(
            CASE WHEN device_rank <= 10 THEN transaction_count ELSE 0 END
        ) / SUM(transaction_count),
        2
    ) AS top_10_pct,
    SUM(CASE WHEN device_rank <= 50 THEN transaction_count ELSE 0 END)
        AS top_50_transactions,
    ROUND(
        100.0 * SUM(
            CASE WHEN device_rank <= 50 THEN transaction_count ELSE 0 END
        ) / SUM(transaction_count),
        2
    ) AS top_50_pct
FROM ranked_devices;
"""

result = con.execute(query).df()
result

,distinct_values,total_transactions,top_10_transactions,top_10_pct,top_50_transactions,top_50_pct
0,1786,118666.0,92162.0,77.67,99574.0,83.91


## Are the common DeviceInfo categories stable across time?

In [36]:
query = """
WITH identity_transactions AS (
    SELECT
        t.TransactionID,
        NTILE(10) OVER (
            ORDER BY t.TransactionDT, t.TransactionID
        ) AS time_group,
        COALESCE(i.DeviceInfo, 'MISSING') AS device_info
    FROM read_parquet('data/interim/train_transaction.parquet') t
    INNER JOIN read_parquet('data/interim/train_identity.parquet') i
        ON t.TransactionID = i.TransactionID
),
device_time_counts AS (
    SELECT
        device_info,
        time_group,
        COUNT(*) AS transaction_count
    FROM identity_transactions
    GROUP BY device_info, time_group
),
ranked_devices AS (
    SELECT
        device_info,
        SUM(transaction_count) AS total_transactions
    FROM device_time_counts
    GROUP BY device_info
    ORDER BY total_transactions DESC
    LIMIT 10
)
SELECT
    d.device_info,
    d.time_group,
    d.transaction_count
FROM device_time_counts d
INNER JOIN ranked_devices r
    ON d.device_info = r.device_info
ORDER BY d.device_info, d.time_group;
"""

result = con.execute(query).df()
result

,device_info,time_group,transaction_count
0,MISSING,1,1896
1,MISSING,2,1653
2,MISSING,3,1607
3,MISSING,4,1748
4,MISSING,5,3125
...,...,...,...
95,rv:57.0,6,43
96,rv:57.0,7,15
97,rv:57.0,8,15
98,rv:57.0,9,6


## 02_sql_data_exploration.ipynb

In [37]:
# Build the training dataset by joining transaction and identity data.
# A LEFT JOIN preserves every transaction, even when identity data is absent.

con.execute("""
CREATE OR REPLACE TEMP VIEW train_model_base AS
SELECT
    t.*,
    i.* EXCLUDE (TransactionID)
FROM read_parquet('data/interim/train_transaction.parquet') AS t
LEFT JOIN read_parquet('data/interim/train_identity.parquet') AS i
USING (TransactionID);
""")

# Verify that the join has not changed the transaction count or target distribution.
result = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT TransactionID) AS unique_transactions,
    COUNT(*) - COUNT(DISTINCT TransactionID) AS duplicate_excess,
    SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
    SUM(CASE WHEN isFraud = 0 THEN 1 ELSE 0 END) AS legitimate_transactions,
    SUM(CASE WHEN DeviceType IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_device_type
FROM train_model_base;
""").df()

display(result)

,total_rows,unique_transactions,duplicate_excess,fraud_transactions,legitimate_transactions,rows_with_device_type
0,590540,590540,0,20663.0,569877.0,140810.0


In [38]:
check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT TransactionID) AS unique_transactions,
    COUNT(*) - COUNT(DISTINCT TransactionID) AS duplicate_excess,
    COUNT(*) FILTER (WHERE isFraud = 1) AS fraud_transactions,
    COUNT(*) FILTER (WHERE isFraud = 0) AS legitimate_transactions
FROM train_model_base;
""").df()

print(check.to_string(index=False))

 total_rows  unique_transactions  duplicate_excess  fraud_transactions  legitimate_transactions
     590540               590540                 0               20663                   569877


## Question: Can we train on earlier transactions and validate on later transactions without mixing the future into the past?

In [39]:
split_check = con.execute("""
WITH ranked AS (
    SELECT
        TransactionID,
        TransactionDT,
        isFraud,
        NTILE(5) OVER (
            ORDER BY TransactionDT, TransactionID
        ) AS time_group
    FROM train_model_base
)
SELECT
    time_group,
    COUNT(*) AS transaction_count,
    MIN(TransactionDT) AS first_transaction_dt,
    MAX(TransactionDT) AS last_transaction_dt,
    SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    ROUND(
        100.0 * SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END)
        / COUNT(*),
        3
    ) AS fraud_rate_pct
FROM ranked
GROUP BY time_group
ORDER BY time_group;
""").df()

print(split_check.to_string(index=False))

 time_group  transaction_count  first_transaction_dt  last_transaction_dt  fraud_count  fraud_rate_pct
          1             118108                 86400              2310138       2826.0           2.393
          2             118108               2310165              5592303       4732.0           4.007
          3             118108               5592304              8745772       4430.0           3.751
          4             118108               8745798             12192842       4611.0           3.904
          5             118108              12192900             15811131       4064.0           3.441


## Before creating the actual datasets, let's confirm the exact transaction-time cutoff and target counts on both sides.

In [40]:
split_summary = con.execute("""
WITH ranked AS (
    SELECT
        TransactionID,
        TransactionDT,
        isFraud,
        NTILE(5) OVER (
            ORDER BY TransactionDT, TransactionID
        ) AS time_group
    FROM train_model_base
)
SELECT
    CASE
        WHEN time_group <= 4 THEN 'train'
        ELSE 'validation'
    END AS dataset_split,
    COUNT(*) AS transaction_count,
    MIN(TransactionDT) AS min_transaction_dt,
    MAX(TransactionDT) AS max_transaction_dt,
    SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
    SUM(CASE WHEN isFraud = 0 THEN 1 ELSE 0 END) AS legitimate_count
FROM ranked
GROUP BY dataset_split
ORDER BY
    CASE WHEN dataset_split = 'train' THEN 1 ELSE 2 END;
""").df()

print(split_summary.to_string(index=False))

dataset_split  transaction_count  min_transaction_dt  max_transaction_dt  fraud_count  legitimate_count
        train             472432               86400            12192842      16599.0          455833.0
   validation             118108            12192900            15811131       4064.0          114044.0


## turn the split into something we can use in later SQL queries and model preparation

In [41]:
# con.execute("""
# CREATE OR REPLACE TEMP VIEW fraudintel_train AS
# WITH ranked AS (
#     SELECT
#         *,
#         NTILE(5) OVER (
#             ORDER BY TransactionDT, TransactionID
#         ) AS time_group
#     FROM train_model_base
# )
# SELECT * EXCLUDE (time_group)
# FROM ranked
# WHERE time_group <= 4;
# """)

# con.execute("""
# CREATE OR REPLACE TEMP VIEW fraudintel_validation AS
# WITH ranked AS (
#     SELECT
#         *,
#         NTILE(5) OVER (
#             ORDER BY TransactionDT, TransactionID
#         ) AS time_group
#     FROM train_model_base
# )
# SELECT * EXCLUDE (time_group)
# FROM ranked
# WHERE time_group = 5;
# """)

# print("Training rows:", con.execute(
#     "SELECT COUNT(*) FROM fraudintel_train"
# ).fetchone()[0])

# print("Validation rows:", con.execute(
#     "SELECT COUNT(*) FROM fraudintel_validation"
# ).fetchone()[0])

In [42]:
con.execute("""
CREATE OR REPLACE TEMP VIEW train_model_base AS
SELECT
    t.*,
    i.* EXCLUDE (TransactionID)
FROM read_parquet('data/interim/train_transaction.parquet') AS t
LEFT JOIN read_parquet('data/interim/train_identity.parquet') AS i
USING (TransactionID);
""")

In [43]:
con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY TransactionDT, TransactionID
    ) AS row_num,
    COUNT(*) OVER () AS total_rows
FROM train_model_base;
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_train AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num <= FLOOR(total_rows * 0.70);
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_validation AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num > FLOOR(total_rows * 0.70)
  AND row_num <= FLOOR(total_rows * 0.85);
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_test AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num > FLOOR(total_rows * 0.85);
""")

print("Training rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_train"
).fetchone()[0])

print("Validation rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_validation"
).fetchone()[0])

print("Test rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_test"
).fetchone()[0])

Training rows: 413378
Validation rows: 88581
Test rows: 88581


In [44]:
split_verification = con.execute("""
SELECT
    'train' AS dataset,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_count,
    COUNT(*) - SUM(isFraud) AS legitimate_count,
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 3) AS fraud_rate_pct,
    MIN(TransactionDT) AS first_transaction_dt,
    MAX(TransactionDT) AS last_transaction_dt
FROM fraudintel_train

UNION ALL

SELECT
    'validation',
    COUNT(*),
    SUM(isFraud),
    COUNT(*) - SUM(isFraud),
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 3),
    MIN(TransactionDT),
    MAX(TransactionDT)
FROM fraudintel_validation

UNION ALL

SELECT
    'test',
    COUNT(*),
    SUM(isFraud),
    COUNT(*) - SUM(isFraud),
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 3),
    MIN(TransactionDT),
    MAX(TransactionDT)
FROM fraudintel_test;
""").df()

display(split_verification)

,dataset,total_transactions,fraud_count,legitimate_count,fraud_rate_pct,first_transaction_dt,last_transaction_dt
0,train,413378,14538.0,398840.0,3.517,86400,10437996
1,validation,88581,3042.0,85539.0,3.434,10438003,13151840
2,test,88581,3083.0,85498.0,3.480,13151880,15811131


In [45]:
con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_test_features AS
SELECT * EXCLUDE (TransactionID, isFraud)
FROM fraudintel_test;
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_test_target AS
SELECT TransactionID, isFraud
FROM fraudintel_test;
""")

print("Test feature columns:",
      len(con.execute("DESCRIBE fraudintel_test_features").df()))

print("Test target rows:",
      con.execute("SELECT COUNT(*) FROM fraudintel_test_target").fetchone()[0])

Test feature columns: 432
Test target rows: 88581


In [46]:
from pathlib import Path

for file in [
    "data/interim/train_transaction.parquet",
    "data/interim/train_identity.parquet",
]:
    path = Path(file)
    print(f"{file}: {'EXISTS' if path.exists() else 'MISSING'}")

data/interim/train_transaction.parquet: EXISTS
data/interim/train_identity.parquet: EXISTS


In [47]:
# 1. Rebuild the joined labelled dataset
con.execute("""
CREATE OR REPLACE TEMP VIEW train_model_base AS
SELECT
    t.*,
    i.* EXCLUDE (TransactionID)
FROM read_parquet('data/interim/train_transaction.parquet') AS t
LEFT JOIN read_parquet('data/interim/train_identity.parquet') AS i
USING (TransactionID);
""")

# 2. Rebuild the chronological ranking
con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY TransactionDT, TransactionID
    ) AS row_num,
    COUNT(*) OVER () AS total_rows
FROM train_model_base;
""")

# 3. Recreate the three datasets
con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_train AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num <= FLOOR(total_rows * 0.70);
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_validation AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num > FLOOR(total_rows * 0.70)
  AND row_num <= FLOOR(total_rows * 0.85);
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW fraudintel_test AS
SELECT * EXCLUDE (row_num, total_rows)
FROM fraudintel_ranked
WHERE row_num > FLOOR(total_rows * 0.85);
""")

# 4. Recreate feature and target views
for split in ["train", "validation", "test"]:
    con.execute(f"""
    CREATE OR REPLACE TEMP VIEW fraudintel_{split}_features AS
    SELECT * EXCLUDE (TransactionID, isFraud)
    FROM fraudintel_{split};
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP VIEW fraudintel_{split}_target AS
    SELECT TransactionID, isFraud
    FROM fraudintel_{split};
    """)

print("PASS: All three splits and six feature/target views recreated.")

for split in ["train", "validation", "test"]:
    rows = con.execute(
        f"SELECT COUNT(*) FROM fraudintel_{split}_features"
    ).fetchone()[0]
    print(f"{split.title()} feature rows: {rows}")

PASS: All three splits and six feature/target views recreated.
Train feature rows: 413378
Validation feature rows: 88581
Test feature rows: 88581


## Save 3 datasets

In [48]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "train_features": "fraudintel_train_features",
    "train_target": "fraudintel_train_target",
    "validation_features": "fraudintel_validation_features",
    "validation_target": "fraudintel_validation_target",
    "test_features": "fraudintel_test_features",
    "test_target": "fraudintel_test_target",
}

for filename, view_name in datasets.items():
    output_path = processed_dir / f"{filename}.parquet"

    con.execute(f"""
        COPY {view_name}
        TO '{output_path.as_posix()}'
        (FORMAT PARQUET);
    """)

    print(f"Saved: {output_path}")

Saved: data\processed\train_features.parquet
Saved: data\processed\train_target.parquet
Saved: data\processed\validation_features.parquet
Saved: data\processed\validation_target.parquet
Saved: data\processed\test_features.parquet
Saved: data\processed\test_target.parquet


In [49]:
con.execute("COPY fraudintel_test_features TO 'data/processed/test_features.parquet' (FORMAT PARQUET)")
con.execute("COPY fraudintel_test_target TO 'data/processed/test_target.parquet' (FORMAT PARQUET)")

print("PASS: Test datasets exported.")

PASS: Test datasets exported.


## audit the model input schema

In [50]:
schema = con.execute("""
DESCRIBE
SELECT *
FROM fraudintel_train;
""").df()

print("Total columns:", len(schema))
print(schema[["column_name", "column_type"]].to_string(index=False))

Total columns: 434
   column_name column_type
 TransactionID      BIGINT
       isFraud      BIGINT
 TransactionDT      BIGINT
TransactionAmt      DOUBLE
     ProductCD     VARCHAR
         card1      BIGINT
         card2      DOUBLE
         card3      DOUBLE
         card4     VARCHAR
         card5      DOUBLE
         card6     VARCHAR
         addr1      DOUBLE
         addr2      DOUBLE
         dist1      DOUBLE
         dist2      DOUBLE
 P_emaildomain     VARCHAR
 R_emaildomain     VARCHAR
            C1      DOUBLE
            C2      DOUBLE
            C3      DOUBLE
            C4      DOUBLE
            C5      DOUBLE
            C6      DOUBLE
            C7      DOUBLE
            C8      DOUBLE
            C9      DOUBLE
           C10      DOUBLE
           C11      DOUBLE
           C12      DOUBLE
           C13      DOUBLE
           C14      DOUBLE
            D1      DOUBLE
            D2      DOUBLE
            D3      DOUBLE
            D4      DOUBLE
         

## build clean modeling views that separate the target from the predictors and exclude the transaction identifier.

In [51]:
print("Training feature columns:",
      len(con.execute("DESCRIBE fraudintel_train_features").df()))

print("Validation feature columns:",
      len(con.execute("DESCRIBE fraudintel_validation_features").df()))

print("Test feature columns:",
      len(con.execute("DESCRIBE fraudintel_test_features").df()))

print("Training target rows:",
      con.execute("SELECT COUNT(*) FROM fraudintel_train_target").fetchone()[0])

print("Validation target rows:",
      con.execute("SELECT COUNT(*) FROM fraudintel_validation_target").fetchone()[0])

print("Test target rows:",
      con.execute("SELECT COUNT(*) FROM fraudintel_test_target").fetchone()[0])

Training feature columns: 432
Validation feature columns: 432
Test feature columns: 432
Training target rows: 413378
Validation target rows: 88581
Test target rows: 88581


## final data-quality check

In [52]:
print("Training rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_train_features"
).fetchone()[0])

print("Validation rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_validation_features"
).fetchone()[0])

print("Test rows:", con.execute(
    "SELECT COUNT(*) FROM fraudintel_test_features"
).fetchone()[0])

Training rows: 413378
Validation rows: 88581
Test rows: 88581


## count numeric and categorical predictors

In [53]:
feature_schema = con.execute("""
DESCRIBE fraudintel_train_features;
""").df()

print("Total predictor columns:", len(feature_schema))
print("\nPredictors by data type:")
print(
    feature_schema.groupby("column_type")
    .size()
    .rename("column_count")
    .to_string()
)

print("\nCategorical predictors:")
print(
    feature_schema.loc[
        feature_schema["column_type"] == "VARCHAR",
        "column_name"
    ].to_list()
)

Total predictor columns: 432

Predictors by data type:
column_type
BIGINT       2
DOUBLE     399
VARCHAR     31

Categorical predictors:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


## create a reproducible modeling dataset

In [54]:
schema_check = con.execute("""
WITH train_schema AS (
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM fraudintel_train_features)
),
validation_schema AS (
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM fraudintel_validation_features)
),
test_schema AS (
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM fraudintel_test_features)
)
SELECT
    COALESCE(t.column_name, v.column_name, x.column_name) AS column_name,
    t.column_type AS train_type,
    v.column_type AS validation_type,
    x.column_type AS test_type
FROM train_schema t
FULL OUTER JOIN validation_schema v USING (column_name)
FULL OUTER JOIN test_schema x USING (column_name)
WHERE t.column_name IS NULL
   OR v.column_name IS NULL
   OR x.column_name IS NULL
   OR t.column_type <> v.column_type
   OR t.column_type <> x.column_type
ORDER BY column_name;
""").df()

print("Schema mismatches:", len(schema_check))
print(schema_check.to_string(index=False) if len(schema_check) else "None")

Schema mismatches: 0
None


## make the dataset reproducible

In [55]:
exports = {
    "train_features.parquet": "fraudintel_train_features",
    "train_target.parquet": "fraudintel_train_target",
    "validation_features.parquet": "fraudintel_validation_features",
    "validation_target.parquet": "fraudintel_validation_target",
    "test_features.parquet": "fraudintel_test_features",
    "test_target.parquet": "fraudintel_test_target",
}

## verify the saved datasets

In [56]:
from pathlib import Path
import duckdb

processed_dir = Path("data/processed")

files = {
    "train_features": processed_dir / "train_features.parquet",
    "train_target": processed_dir / "train_target.parquet",
    "validation_features": processed_dir / "validation_features.parquet",
    "validation_target": processed_dir / "validation_target.parquet",
    "test_features": processed_dir / "test_features.parquet",
    "test_target": processed_dir / "test_target.parquet",
}

for name, path in files.items():
    print(f"{name}: exists={path.exists()}, size={path.stat().st_size:,} bytes")

con = duckdb.connect()

for name, path in files.items():
    result = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(path)]
    ).fetchone()[0]
    print(f"{name} rows: {result:,}")

for split in ["train", "validation", "test"]:
    print(f"\n{split.title()} target distribution:")
    print(con.execute(f"""
        SELECT isFraud, COUNT(*) AS transactions
        FROM read_parquet('data/processed/{split}_target.parquet')
        GROUP BY isFraud
        ORDER BY isFraud;
    """).df().to_string(index=False))

train_features: exists=True, size=55,158,170 bytes
train_target: exists=True, size=1,686,210 bytes
validation_features: exists=True, size=11,342,457 bytes
validation_target: exists=True, size=361,420 bytes
test_features: exists=True, size=12,524,157 bytes
test_target: exists=True, size=361,474 bytes
train_features rows: 413,378
train_target rows: 413,378
validation_features rows: 88,581
validation_target rows: 88,581
test_features rows: 88,581
test_target rows: 88,581

Train target distribution:
 isFraud  transactions
       0        398840
       1         14538

Validation target distribution:
 isFraud  transactions
       0         85539
       1          3042

Test target distribution:
 isFraud  transactions
       0         85498
       1          3083


## preprocessing and baseline model

In [57]:
from pathlib import Path
import pandas as pd

# Locate the processed datasets
PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load training, validation and test datasets
X_train = pd.read_parquet(PROCESSED_DIR / "train_features.parquet")
train_target = pd.read_parquet(PROCESSED_DIR / "train_target.parquet")

X_val = pd.read_parquet(PROCESSED_DIR / "validation_features.parquet")
val_target = pd.read_parquet(PROCESSED_DIR / "validation_target.parquet")

X_test = pd.read_parquet(PROCESSED_DIR / "test_features.parquet")
test_target = pd.read_parquet(PROCESSED_DIR / "test_target.parquet")

# Extract target labels
y_train = train_target["isFraud"]
y_val = val_target["isFraud"]
y_test = test_target["isFraud"]

# Inspect datasets
print("TRAINING DATA")
print("Features:", X_train.shape)
print("Target:", y_train.shape)
print(y_train.value_counts().sort_index())

print("\nVALIDATION DATA")
print("Features:", X_val.shape)
print("Target:", y_val.shape)
print(y_val.value_counts().sort_index())

print("\nTEST DATA")
print("Features:", X_test.shape)
print("Target:", y_test.shape)
print(y_test.value_counts().sort_index())

print("\nFEATURE SCHEMA")
print("Train vs validation:", X_train.columns.equals(X_val.columns))
print("Train vs test:", X_train.columns.equals(X_test.columns))

print("\nMISSING VALUES")
print("Training:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

print("\nTARGET VALIDATION")
print("Training missing labels:", y_train.isna().sum())
print("Validation missing labels:", y_val.isna().sum())
print("Test missing labels:", y_test.isna().sum())

print("Training labels:", sorted(y_train.dropna().unique()))
print("Validation labels:", sorted(y_val.dropna().unique()))
print("Test labels:", sorted(y_test.dropna().unique()))

TRAINING DATA
Features: (413378, 432)
Target: (413378,)
isFraud
0    398840
1     14538
Name: count, dtype: int64

VALIDATION DATA
Features: (88581, 432)
Target: (88581,)
isFraud
0    85539
1     3042
Name: count, dtype: int64

TEST DATA
Features: (88581, 432)
Target: (88581,)
isFraud
0    85498
1     3083
Name: count, dtype: int64

FEATURE SCHEMA
Train vs validation: True
Train vs test: True

MISSING VALUES
Training: 80397089
Validation: 17913146
Test: 17212838

TARGET VALIDATION
Training missing labels: 0
Validation missing labels: 0
Test missing labels: 0
Training labels: [np.int64(0), np.int64(1)]
Validation labels: [np.int64(0), np.int64(1)]
Test labels: [np.int64(0), np.int64(1)]


## Verifying alignment

In [58]:

# Check which DuckDB views are currently available
print("Available DuckDB views:")

print(
    con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_type = 'VIEW'
        ORDER BY table_name
    """).df().to_string(index=False)
)


Available DuckDB views:
Empty DataFrame
Columns: [table_name]
Index: []


##Let's check whether the notebook has another DuckDB connection variable that might still contain the original views.

In [59]:

# Find DuckDB connection variables currently available in the notebook
[
    name
    for name, value in globals().items()
    if "duckdb" in type(value).__module__.lower()
    or type(value).__name__ == "DuckDBPyConnection"
]


['_', 'con', '_42']

In [60]:

# Inspect the target files and their transaction IDs

print("TRAIN TARGET")
print(train_target.head(10).to_string(index=False))
print("Duplicate training IDs:", train_target["TransactionID"].duplicated().sum())
print("Null training IDs:", train_target["TransactionID"].isna().sum())

print("\nVALIDATION TARGET")
print(val_target.head(10).to_string(index=False))
print("Duplicate validation IDs:", val_target["TransactionID"].duplicated().sum())
print("Null validation IDs:", val_target["TransactionID"].isna().sum())

print("\nTEST TARGET")
print(test_target.head(10).to_string(index=False))
print("Duplicate test IDs:", test_target["TransactionID"].duplicated().sum())
print("Null test IDs:", test_target["TransactionID"].isna().sum())

TRAIN TARGET
 TransactionID  isFraud
       2987000        0
       2987001        0
       2987002        0
       2987003        0
       2987004        0
       2987005        0
       2987006        0
       2987007        0
       2987008        0
       2987009        0
Duplicate training IDs: 0
Null training IDs: 0

VALIDATION TARGET
 TransactionID  isFraud
       3400378        0
       3400379        1
       3400380        0
       3400381        0
       3400382        0
       3400383        0
       3400384        0
       3400385        0
       3400386        0
       3400387        0
Duplicate validation IDs: 0
Null validation IDs: 0

TEST TARGET
 TransactionID  isFraud
       3488959        0
       3488960        0
       3488961        0
       3488962        0
       3488963        0
       3488964        0
       3488965        0
       3488966        0
       3488967        0
       3488968        0
Duplicate test IDs: 0
Null test IDs: 0


## Rebuild the aligned datasets

In [61]:
print("TRAIN")
print("Rows:", len(X_train))
print("Unique IDs:", train_target["TransactionID"].nunique())
print("Fraud cases:", int(y_train.sum()))

print("\nVALIDATION")
print("Rows:", len(X_val))
print("Unique IDs:", val_target["TransactionID"].nunique())
print("Fraud cases:", int(y_val.sum()))

print("\nTEST")
print("Rows:", len(X_test))
print("Unique IDs:", test_target["TransactionID"].nunique())
print("Fraud cases:", int(y_test.sum()))

print("\nPASS: All three chronological splits are loaded.")

TRAIN
Rows: 413378
Unique IDs: 413378
Fraud cases: 14538

VALIDATION
Rows: 88581
Unique IDs: 88581
Fraud cases: 3042

TEST
Rows: 88581
Unique IDs: 88581
Fraud cases: 3083

PASS: All three chronological splits are loaded.


## train our first CatBoost baseline

In [62]:

# from pathlib import Path
# import json
# import time

# from catboost import CatBoostClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     average_precision_score,
#     roc_auc_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     confusion_matrix,
#     classification_report,
# )

# # 1. Prepare categorical features
# X_train_cat = X_train.copy()
# X_val_cat = X_val.copy()

# categorical_columns = X_train_cat.select_dtypes(
#     include=["object", "category", "string"]
# ).columns.tolist()

# for column in categorical_columns:
#     X_train_cat[column] = (
#         X_train_cat[column]
#         .astype("string")
#         .fillna("__MISSING__")
#         .astype(str)
#     )

#     X_val_cat[column] = (
#         X_val_cat[column]
#         .astype("string")
#         .fillna("__MISSING__")
#         .astype(str)
#     )

# # 2. Select stratified 20% training sample
# X_train_cat_sample, _, y_train_cat_sample, _ = train_test_split(
#     X_train_cat,
#     y_train,
#     train_size=0.20,
#     random_state=42,
#     stratify=y_train,
# )

# print("Training sample:", len(X_train_cat_sample))
# print("Validation rows:", len(X_val_cat))
# print("Features:", X_train_cat_sample.shape[1])
# print("Categorical features:", len(categorical_columns))

# # 3. Configure CatBoost
# model = CatBoostClassifier(
#     iterations=100,
#     depth=4,
#     learning_rate=0.1,
#     loss_function="Logloss",
#     eval_metric="PRAUC",
#     random_seed=42,
#     verbose=10,
#     allow_writing_files=False,
#     thread_count=-1,
# )

# # 4. Train with visible progress
# print("\n===== CATBOOST TRAINING STARTED =====")

# start_time = time.time()

# model.fit(
#     X_train_cat_sample,
#     y_train_cat_sample,
#     cat_features=categorical_columns,
#     eval_set=(X_val_cat, y_val),
#     early_stopping_rounds=20,
#     use_best_model=True,
#     verbose=10,
# )

# training_time = round(time.time() - start_time, 2)

# print("\n===== TRAINING COMPLETED =====")
# print(f"Training time: {training_time:.2f} seconds")
# print("Best iteration:", model.get_best_iteration())

# # 5. Validation predictions
# y_prob = model.predict_proba(X_val_cat)[:, 1]
# y_pred = (y_prob >= 0.5).astype(int)

# # 6. Evaluate performance
# metrics = {
#     "model": "CatBoost",
#     "training_sample_fraction": 0.20,
#     "threshold": 0.5,
#     "average_precision": float(average_precision_score(y_val, y_prob)),
#     "roc_auc": float(roc_auc_score(y_val, y_prob)),
#     "precision": float(precision_score(y_val, y_pred, zero_division=0)),
#     "recall": float(recall_score(y_val, y_pred, zero_division=0)),
#     "f1": float(f1_score(y_val, y_pred, zero_division=0)),
#     "best_iteration": int(model.get_best_iteration()),
#     "training_time_seconds": training_time,
#     "training_rows": len(X_train_cat_sample),
#     "validation_rows": len(X_val_cat),
# }

# print("\n===== CATBOOST VALIDATION METRICS =====")
# for name, value in metrics.items():
#     print(f"{name}: {value}")

# print("\n===== CONFUSION MATRIX =====")
# print(confusion_matrix(y_val, y_pred))

# print("\n===== CLASSIFICATION REPORT =====")
# print(classification_report(
#     y_val,
#     y_pred,
#     target_names=["Legitimate", "Fraud"],
#     zero_division=0,
# ))

# # 7. Save model and metrics
# output_dir = Path("artifacts/baseline_catboost")
# output_dir.mkdir(parents=True, exist_ok=True)

# model.save_model(str(output_dir / "model.cbm"))

# with open(output_dir / "metrics.json", "w", encoding="utf-8") as file:
#     json.dump(metrics, file, indent=4)

# print("\nModel saved:", output_dir / "model.cbm")
# print("Metrics saved:", output_dir / "metrics.json")
# print("PASS: CatBoost diagnostic training completed.")


In [63]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.51, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(
            y_val, y_pred_threshold, zero_division=0
        ),
        "recall": recall_score(
            y_val, y_pred_threshold, zero_division=0
        ),
        "f1": f1_score(
            y_val, y_pred_threshold, zero_division=0
        ),
    })

threshold_results = pd.DataFrame(threshold_results)

print("===== CATBOOST THRESHOLD ANALYSIS =====")
print(threshold_results.to_string(index=False))

best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print("\n===== BEST F1 THRESHOLD =====")
print(best_f1_row)

NameError: name 'y_prob' is not defined

## LightGBM baseline

In [65]:
selected_features = [
    "TransactionAmt",
    "TransactionDT",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "dist1",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "id_31",
]

print(f"Selected features: {len(selected_features)}")

Selected features: 17


In [66]:
from pathlib import Path
import json
import time

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# 1. Prepare data
#X_train_lgb = X_train.copy()
#X_val_lgb = X_val.copy()

X_train_lgb = X_train[selected_features].copy()
X_val_lgb = X_val[selected_features].copy()

# Convert categorical columns to category dtype
categorical_columns = X_train_lgb.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for column in categorical_columns:
    X_train_lgb[column] = (
        X_train_lgb[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype("category")
    )

    X_val_lgb[column] = (
        X_val_lgb[column]
        .astype("string")
        .fillna("__MISSING__")
        .astype("category")
    )

print("Training rows:", len(X_train_lgb))
print("Validation rows:", len(X_val_lgb))
print("Features:", X_train_lgb.shape[1])
print("Categorical features:", len(categorical_columns))

# 2. Train LightGBM
model_lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

print("\n===== LIGHTGBM TRAINING STARTED =====")

start_time = time.time()

model_lgb.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_columns,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="average_precision",
    callbacks=[
        early_stopping(30, verbose=True),
        log_evaluation(10),
    ],
)

training_time = round(time.time() - start_time, 2)

print("\n===== TRAINING COMPLETED =====")
print(f"Training time: {training_time:.2f} seconds")
print("Best iteration:", model_lgb.best_iteration_)

# 3. Validation predictions
y_prob_lgb = model_lgb.predict_proba(
    X_val_lgb,
    num_iteration=model_lgb.best_iteration_
)[:, 1]

y_pred_lgb = (y_prob_lgb >= 0.5).astype(int)

# 4. Evaluate baseline
metrics_lgb = {
    "model": "LightGBM",
    "threshold": 0.5,
    "average_precision": float(
        average_precision_score(y_val, y_prob_lgb)
    ),
    "roc_auc": float(
        roc_auc_score(y_val, y_prob_lgb)
    ),
    "precision": float(
        precision_score(y_val, y_pred_lgb, zero_division=0)
    ),
    "recall": float(
        recall_score(y_val, y_pred_lgb, zero_division=0)
    ),
    "f1": float(
        f1_score(y_val, y_pred_lgb, zero_division=0)
    ),
    "best_iteration": int(model_lgb.best_iteration_),
    "training_time_seconds": training_time,
    "training_rows": len(X_train_lgb),
    "validation_rows": len(X_val_lgb),
    "predictor_columns": X_train_lgb.shape[1],
    "categorical_columns": len(categorical_columns),
}

print("\n===== LIGHTGBM BASELINE METRICS =====")
for name, value in metrics_lgb.items():
    print(f"{name}: {value}")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_val, y_pred_lgb))

print("\n===== CLASSIFICATION REPORT =====")
print(
    classification_report(
        y_val,
        y_pred_lgb,
        target_names=["Legitimate", "Fraud"],
        zero_division=0,
    )
)

# 5. Save model and metrics
output_dir = Path("artifacts/baseline_lightgbm")
output_dir.mkdir(parents=True, exist_ok=True)

model_lgb.booster_.save_model(
    str(output_dir / "model.txt")
)

with open(
    output_dir / "metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(metrics_lgb, file, indent=4)

print("\nModel saved:", output_dir / "model.txt")
print("Metrics saved:", output_dir / "metrics.json")
print("PASS: LightGBM baseline training completed.")

Training rows: 413378
Validation rows: 88581
Features: 17
Categorical features: 8

===== LIGHTGBM TRAINING STARTED =====


C:\Users\NMP\PycharmProjects\FraudIntel\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 30 rounds
[10]	valid_0's average_precision: 0.195765	valid_0's binary_logloss: 0.133506
[20]	valid_0's average_precision: 0.207015	valid_0's binary_logloss: 0.129194
[30]	valid_0's average_precision: 0.212414	valid_0's binary_logloss: 0.126908
[40]	valid_0's average_precision: 0.216762	valid_0's binary_logloss: 0.125428
[50]	valid_0's average_precision: 0.223608	valid_0's binary_logloss: 0.12388
[60]	valid_0's average_precision: 0.231254	valid_0's binary_logloss: 0.12268
[70]	valid_0's average_precision: 0.237739	valid_0's binary_logloss: 0.121729
[80]	valid_0's average_precision: 0.241026	valid_0's binary_logloss: 0.121219
[90]	valid_0's average_precision: 0.246403	valid_0's binary_logloss: 0.12045
[100]	valid_0's average_precision: 0.249925	valid_0's binary_logloss: 0.120062
[110]	valid_0's average_precision: 0.252542	valid_0's binary_logloss: 0.119795
[120]	valid_0's average_precision: 0.25594	valid_0's binary_logloss: 0.119452
[130

## Threshold Analysis

In [67]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.01, 0.51, 0.01)

threshold_results_lgb = []

for threshold in thresholds:
    y_pred_threshold = (y_prob_lgb >= threshold).astype(int)

    threshold_results_lgb.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(
            y_val,
            y_pred_threshold,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            y_pred_threshold,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            y_pred_threshold,
            zero_division=0
        ),
    })

threshold_results_lgb = pd.DataFrame(threshold_results_lgb)

print("===== LIGHTGBM THRESHOLD ANALYSIS =====")
print(threshold_results_lgb.to_string(index=False))

best_f1_row_lgb = threshold_results_lgb.loc[
    threshold_results_lgb["f1"].idxmax()
]

print("\n===== BEST F1 THRESHOLD =====")
print(best_f1_row_lgb)

===== LIGHTGBM THRESHOLD ANALYSIS =====
 threshold  precision   recall       f1
      0.01   0.046997 0.939842 0.089517
      0.02   0.077373 0.813281 0.141303
      0.03   0.105530 0.718935 0.184044
      0.04   0.133005 0.657462 0.221251
      0.05   0.157078 0.605851 0.249475
      0.06   0.182447 0.571992 0.276652
      0.07   0.203615 0.529586 0.294139
      0.08   0.223436 0.497699 0.308413
      0.09   0.242100 0.476003 0.320958
      0.10   0.262179 0.463511 0.334917
      0.11   0.276136 0.445431 0.340923
      0.12   0.292552 0.429980 0.348196
      0.13   0.310064 0.413215 0.354284
      0.14   0.323632 0.398422 0.357153
      0.15   0.334202 0.379356 0.355350
      0.16   0.342419 0.359303 0.350658
      0.17   0.352239 0.349112 0.350669
      0.18   0.366287 0.338593 0.351896
      0.19   0.378409 0.328402 0.351637
      0.20   0.387896 0.313938 0.347020
      0.21   0.393534 0.300131 0.340545
      0.22   0.402544 0.291256 0.337974
      0.23   0.413219 0.285667 0.337804


## recall-constrained threshold selection

In [68]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

# Evaluate lower LightGBM thresholds
thresholds = np.round(np.arange(0.01, 0.051, 0.005), 3)

y_true_lgb = np.asarray(y_val)

results_lgb = []

for threshold in thresholds:

    y_pred = (y_prob_lgb >= threshold).astype(int)

    results_lgb.append({
        "threshold": threshold,
        "precision": precision_score(
            y_true_lgb, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true_lgb, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true_lgb, y_pred, zero_division=0
        )
    })

recall_threshold_results_lgb = pd.DataFrame(results_lgb)

# Select highest precision while maintaining at least 70% recall
eligible = recall_threshold_results_lgb[
    recall_threshold_results_lgb["recall"] >= 0.70
]

if not eligible.empty:

    best_recall_row = eligible.sort_values(
        ["precision", "recall"],
        ascending=False
    ).iloc[0]

    print("===== LIGHTGBM RECALL-OPTIMIZED THRESHOLD =====")
    print(best_recall_row)

else:
    print("No tested threshold achieved the 70% recall target.")
    print(
        "Maximum recall:",
        recall_threshold_results_lgb["recall"].max()
    )

# Save results
report_path = Path("reports/lightgbm_recall_threshold_analysis.csv")
report_path.parent.mkdir(parents=True, exist_ok=True)

recall_threshold_results_lgb.to_csv(report_path, index=False)

assert report_path.exists()

print("\nPASS: Recall-oriented threshold analysis saved.")
display(recall_threshold_results_lgb)

===== LIGHTGBM RECALL-OPTIMIZED THRESHOLD =====
threshold    0.030000
precision    0.105530
recall       0.718935
f1           0.184044
Name: 4, dtype: float64

PASS: Recall-oriented threshold analysis saved.


,threshold,precision,recall,f1
0,0.010,0.046997,0.939842,0.089517
1,0.015,0.062911,0.874753,0.117380
2,0.020,0.077373,0.813281,0.141303
3,0.025,0.091529,0.763314,0.163458
4,0.030,0.105530,0.718935,0.184044
5,0.035,0.118304,0.682446,0.201651
6,0.040,0.133005,0.657462,0.221251
7,0.045,0.145635,0.632807,0.236777
8,0.050,0.157078,0.605851,0.249475


## CatBoost recall-oriented threshold analysis

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score

# CatBoost validation probabilities
y_prob_cat = np.asarray(y_prob)

# Validation labels
y_true_cat = np.asarray(y_val)

# Same thresholds used for LightGBM
thresholds = np.round(np.arange(0.01, 0.051, 0.005), 3)

results_cat = []

for threshold in thresholds:

    y_pred = (y_prob_cat >= threshold).astype(int)

    results_cat.append({
        "threshold": threshold,
        "precision": precision_score(
            y_true_cat, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true_cat, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true_cat, y_pred, zero_division=0
        )
    })

catboost_recall_results = pd.DataFrame(results_cat)

# Select highest precision with at least 70% recall
eligible = catboost_recall_results[
    catboost_recall_results["recall"] >= 0.70
]

if not eligible.empty:

    best_catboost_row = eligible.sort_values(
        ["precision", "recall"],
        ascending=False
    ).iloc[0]

    print("===== CATBOOST RECALL-OPTIMIZED THRESHOLD =====")
    print(best_catboost_row)

else:
    print("No threshold achieved the 70% recall target.")
    print(
        "Maximum recall:",
        catboost_recall_results["recall"].max()
    )

# Save report
report_path = Path(
    "reports/catboost_recall_threshold_analysis.csv"
)

report_path.parent.mkdir(parents=True, exist_ok=True)

catboost_recall_results.to_csv(report_path, index=False)

assert report_path.exists()

saved_catboost_results = pd.read_csv(report_path)

assert len(saved_catboost_results) == len(catboost_recall_results)

print("\nPASS: CatBoost recall-oriented threshold report saved and verified.")

display(saved_catboost_results)

## save the model comparison

In [73]:
from pathlib import Path
import pandas as pd

lightgbm_best = (
    recall_threshold_results_lgb[
        recall_threshold_results_lgb["recall"] >= 0.70
    ]
    .sort_values("precision", ascending=False)
    .iloc[0]
)

catboost_best = (
    catboost_recall_results[
        catboost_recall_results["recall"] >= 0.70
    ]
    .sort_values("precision", ascending=False)
    .iloc[0]
)

comparison = pd.DataFrame([
    {
        "model": "LightGBM",
        **lightgbm_best.to_dict()
    },
    {
        "model": "CatBoost",
        **catboost_best.to_dict()
    }
])

comparison = comparison[
    ["model", "threshold", "precision", "recall", "f1"]
]

comparison_path = Path(
    "reports/recall_optimized_model_comparison.csv"
)

comparison_path.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(comparison_path, index=False)

assert comparison_path.exists()
assert len(comparison) == 2
assert (comparison["recall"] >= 0.70).all()

print("PASS: Recall-optimized model comparison saved.")
display(comparison)

PASS: Recall-optimized model comparison saved.


,model,threshold,precision,recall,f1
0,LightGBM,0.050,0.251318,0.736358,0.374739
1,CatBoost,0.035,0.155139,0.701183,0.254065


## validation alignment check

In [74]:
import numpy as np

# Confirm validation labels and prediction arrays
y_true = np.asarray(y_val)
lgb_probs = np.asarray(y_prob_lgb)
cat_probs = np.asarray(y_prob_cat)

# 1. Check array lengths
assert len(y_true) == len(lgb_probs), \
    "LightGBM predictions do not match validation labels."

assert len(y_true) == len(cat_probs), \
    "CatBoost predictions do not match validation labels."

# 2. Check probability validity
assert np.isfinite(lgb_probs).all()
assert np.isfinite(cat_probs).all()

assert ((lgb_probs >= 0) & (lgb_probs <= 1)).all()
assert ((cat_probs >= 0) & (cat_probs <= 1)).all()

# 3. Check target classes
assert set(np.unique(y_true)) == {0, 1}

print("===== VALIDATION ALIGNMENT CHECK =====")
print("Validation records:", len(y_true))
print("Fraud cases:", int(y_true.sum()))
print("Non-fraud cases:", int((y_true == 0).sum()))
print("LightGBM predictions:", len(lgb_probs))
print("CatBoost predictions:", len(cat_probs))

print("\nPASS: Prediction lengths and validation labels verified.")

===== VALIDATION ALIGNMENT CHECK =====
Validation records: 88581
Fraud cases: 3042
Non-fraud cases: 85539
LightGBM predictions: 88581
CatBoost predictions: 88581

PASS: Prediction lengths and validation labels verified.


## confirm validation data alignment

In [75]:
print("TEST DATA VARIABLES")

for name in [
    "X_test",
    "y_test",
    "X_test_lgb",
    "y_test_lgb",
    "X_test_cat",
    "y_test_cat",
]:
    if name in globals():
        print(name, globals()[name].shape)

TEST DATA VARIABLES
X_test (88581, 432)
y_test (88581,)


## LightGBM final test evaluation

In [83]:
print("===== UNSEEN VALIDATION CATEGORIES =====")

for col in cat_cols:
    unseen_count = X_val[col].isna().sum()
    if unseen_count > 0:
        print(f"{col}: {unseen_count:,} missing values")

print("\nTraining shape:", X_train.shape)
print("Validation shape:", X_val.shape)

===== UNSEEN VALIDATION CATEGORIES =====
card4: 4 missing values
card6: 2 missing values
P_emaildomain: 14,130 missing values
R_emaildomain: 72,849 missing values
M1: 23,724 missing values
M2: 23,724 missing values
M3: 23,724 missing values
M4: 40,817 missing values
M5: 50,415 missing values
M6: 19,001 missing values
M7: 35,407 missing values
M8: 35,399 missing values
M9: 35,399 missing values
id_12: 72,927 missing values
id_15: 73,338 missing values
id_16: 74,766 missing values
id_23: 87,996 missing values
id_27: 87,996 missing values
id_28: 73,338 missing values
id_29: 73,338 missing values
id_30: 81,833 missing values
id_31: 73,384 missing values
id_33: 81,921 missing values
id_34: 81,826 missing values
id_35: 73,338 missing values
id_36: 73,338 missing values
id_37: 73,338 missing values
id_38: 73,338 missing values
DeviceType: 73,350 missing values
DeviceInfo: 76,656 missing values

Training shape: (413378, 432)
Validation shape: (88581, 432)


## Checking Available Categorical Columns

In [85]:
# Identify categorical columns from training data
cat_cols = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# Standardise categorical values
for col in cat_cols:
    X_train[col] = (
        X_train[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    X_val[col] = (
        X_val[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    # Categories learned from training
    train_categories = set(X_train[col].unique())

    # Map unseen validation values to a known category
    X_val.loc[
        ~X_val[col].isin(train_categories), col
    ] = "__UNKNOWN__"

    # Add unknown category to training and validation
    categories = sorted(
        train_categories | {"__UNKNOWN__"}
    )

    X_train[col] = pd.Categorical(
        X_train[col], categories=categories
    )

    X_val[col] = pd.Categorical(
        X_val[col], categories=categories
    )

print("PASS: Categorical preprocessing completed.")
print("Categorical columns:", len(cat_cols))
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

PASS: Categorical preprocessing completed.
Categorical columns: 31
Training shape: (413378, 432)
Validation shape: (88581, 432)


## Train LightGBM with validation

In [87]:
# Retrain the existing LightGBM model
model_lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols
)

print("PASS: LightGBM training completed.")

# Generate validation probabilities
y_prob_lgb = model_lgb.predict_proba(X_val)[:, 1]

print("Validation predictions:", len(y_prob_lgb))
print("Fraud probabilities generated successfully.")

C:\Users\NMP\PycharmProjects\FraudIntel\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


PASS: LightGBM training completed.
Validation predictions: 88581
Fraud probabilities generated successfully.


## Reassess the recall-oriented threshold using these newly generated validation probabilities.

In [88]:
from sklearn.metrics import precision_score, recall_score, f1_score


thresholds = [0.01, 0.015, 0.02, 0.025, 0.03,
              0.035, 0.04, 0.045, 0.05]

results = []

for threshold in thresholds:
    predictions = (y_prob_lgb >= threshold).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, predictions),
        "recall": recall_score(y_val, predictions),
        "f1": f1_score(y_val, predictions)
    })

lightgbm_threshold_results = pd.DataFrame(results)

display(lightgbm_threshold_results.round(4))

,threshold,precision,recall,f1
0,0.010,0.0758,0.9359,0.1402
1,0.015,0.1037,0.8951,0.1859
2,0.020,0.1285,0.8596,0.2235
3,0.025,0.1510,0.8261,0.2553
4,0.030,0.1728,0.8037,0.2844
5,0.035,0.1940,0.7853,0.3111
6,0.040,0.2124,0.7663,0.3326
7,0.045,0.2320,0.7515,0.3545
8,0.050,0.2513,0.7364,0.3747


## Save the retrained LightGBM model and its selected threshold

In [89]:
from pathlib import Path
import joblib

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model_lgb, model_dir / "lightgbm_model.joblib")

threshold_config = {
    "model": "LightGBM",
    "threshold": 0.05,
    "precision": 0.2513,
    "recall": 0.7364,
    "f1": 0.3747
}

joblib.dump(threshold_config, model_dir / "lightgbm_threshold.joblib")

print("PASS: LightGBM model and threshold saved.")

PASS: LightGBM model and threshold saved.


## Business Cost Analysis

In [93]:

import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix

# ============================================================
# 1. BUSINESS COST ASSUMPTIONS
# ============================================================

COST_PER_MISSED_FRAUD = 100.0
COST_PER_FALSE_ALARM = 5.0
MIN_RECALL = 0.70

# ============================================================
# 2. EXISTING MODEL PROBABILITIES
# ============================================================

model_probabilities = {
    "CatBoost": y_prob,
    "LightGBM": y_prob_lgb
}

# Use the same threshold values for both models
thresholds = sorted(threshold_results["threshold"].unique())

# ============================================================
# 3. CALCULATE BUSINESS COSTS FOR BOTH MODELS
# ============================================================

results = []

for model_name, probabilities in model_probabilities.items():

    for threshold in thresholds:

        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val,
            predictions,
            labels=[0, 1]
        ).ravel()

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) else 0
        )

        missed_fraud_cost = fn * COST_PER_MISSED_FRAUD
        false_alarm_cost = fp * COST_PER_FALSE_ALARM

        total_cost = missed_fraud_cost + false_alarm_cost

        results.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "true_positives": tp,
            "false_positives": fp,
            "false_negatives": fn,
            "true_negatives": tn,
            "missed_fraud_cost": missed_fraud_cost,
            "false_alarm_cost": false_alarm_cost,
            "total_estimated_cost": total_cost,
            "cost_per_validation_transaction": total_cost / len(y_val)
        })

cost_results = pd.DataFrame(results)

# ============================================================
# 4. DISPLAY COMPLETE BUSINESS COST COMPARISON
# ============================================================

print("========== BUSINESS COST COMPARISON ==========")

display(
    cost_results.round(4)
)

# ============================================================
# 5. APPLY MINIMUM RECALL REQUIREMENT
# ============================================================

eligible_results = cost_results[
    cost_results["recall"] >= MIN_RECALL
].copy()

if eligible_results.empty:
    raise ValueError(
        "No model-threshold combination meets the minimum recall requirement."
    )

# ============================================================
# 6. LOWEST ESTIMATED COST FOR EACH MODEL
# ============================================================

best_cost_by_model = (
    eligible_results.loc[
        eligible_results.groupby("model")[
            "total_estimated_cost"
        ].idxmin()
    ]
    .sort_values("total_estimated_cost")
)

print("\n========== LOWEST COST BY MODEL ==========")
print(f"Minimum recall requirement: {MIN_RECALL:.0%}")

display(
    best_cost_by_model[
        [
            "model",
            "threshold",
            "precision",
            "recall",
            "f1",
            "true_positives",
            "false_positives",
            "false_negatives",
            "total_estimated_cost",
            "cost_per_validation_transaction"
        ]
    ].round(4)
)

# ============================================================
# 7. IDENTIFY OVERALL LOWEST-COST MODEL
# ============================================================

best_overall = eligible_results.loc[
    eligible_results["total_estimated_cost"].idxmin()
]

print("\n========== OVERALL BEST BUSINESS DECISION ==========")
print(f"Model: {best_overall['model']}")
print(f"Threshold: {best_overall['threshold']:.3f}")
print(f"Precision: {best_overall['precision']:.4f}")
print(f"Recall: {best_overall['recall']:.4f}")
print(f"F1-score: {best_overall['f1']:.4f}")
print(f"True positives: {best_overall['true_positives']}")
print(f"False positives: {best_overall['false_positives']}")
print(f"False negatives: {best_overall['false_negatives']}")
print(
    f"Estimated total cost: "
    f"${best_overall['total_estimated_cost']:,.2f}"
)
print(
    f"Cost per validation transaction: "
    f"${best_overall['cost_per_validation_transaction']:.4f}"
)

# ============================================================
# 8. SAVE COMPLETE BUSINESS COST REPORT
# ============================================================

report_dir = Path("reports")
report_dir.mkdir(parents=True, exist_ok=True)

cost_report_path = (
    report_dir / "fraud_business_cost_analysis.csv"
)

cost_results.to_csv(cost_report_path, index=False)

# Save best eligible model-threshold combinations separately
best_cost_path = (
    report_dir / "best_business_cost_by_model.csv"
)

best_cost_by_model.to_csv(best_cost_path, index=False)

print("\n========== REPORTS SAVED ==========")
print(f"Full comparison: {cost_report_path}")
print(f"Best by model: {best_cost_path}")
print("\nPASS: Business cost analysis completed for both models.")


========== BUSINESS COST COMPARISON ==========


,model,threshold,precision,recall,f1,true_positives,false_positives,false_negatives,true_negatives,missed_fraud_cost,false_alarm_cost,total_estimated_cost,cost_per_validation_transaction
0,CatBoost,0.05,0.2071,0.6364,0.3125,1936,7413,1106,78126,110600.0,37065.0,147665.0,1.6670
1,CatBoost,0.06,0.2425,0.5980,0.3450,1819,5683,1223,79856,122300.0,28415.0,150715.0,1.7014
2,CatBoost,0.07,0.2777,0.5680,0.3731,1728,4494,1314,81045,131400.0,22470.0,153870.0,1.7371
3,CatBoost,0.08,0.3132,0.5371,0.3957,1634,3583,1408,81956,140800.0,17915.0,158715.0,1.7917
4,CatBoost,0.09,0.3450,0.5171,0.4138,1573,2987,1469,82552,146900.0,14935.0,161835.0,1.8270
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,LightGBM,0.46,0.8023,0.3734,0.5096,1136,280,1906,85259,190600.0,1400.0,192000.0,2.1675
88,LightGBM,0.47,0.8075,0.3682,0.5058,1120,267,1922,85272,192200.0,1335.0,193535.0,2.1848
89,LightGBM,0.48,0.8114,0.3649,0.5034,1110,258,1932,85281,193200.0,1290.0,194490.0,2.1956
90,LightGBM,0.49,0.8170,0.3596,0.4994,1094,245,1948,85294,194800.0,1225.0,196025.0,2.2129



========== LOWEST COST BY MODEL ==========
Minimum recall requirement: 70%


,model,threshold,precision,recall,f1,true_positives,false_positives,false_negatives,total_estimated_cost,cost_per_validation_transaction
46,LightGBM,0.05,0.2513,0.7364,0.3747,2240,6673,802,113565.0,1.282



========== OVERALL BEST BUSINESS DECISION ==========
Model: LightGBM
Threshold: 0.050
Precision: 0.2513
Recall: 0.7364
F1-score: 0.3747
True positives: 2240
False positives: 6673
False negatives: 802
Estimated total cost: $113,565.00
Cost per validation transaction: $1.2820

========== REPORTS SAVED ==========
Full comparison: reports\fraud_business_cost_analysis.csv
Best by model: reports\best_business_cost_by_model.csv

PASS: Business cost analysis completed for both models.


In [94]:
print("MAXIMUM RECALL BY MODEL")
print(cost_results.groupby("model")["recall"].max())

print("\nBEST ELIGIBLE RESULTS")
print(best_cost_by_model.to_string(index=False))

MAXIMUM RECALL BY MODEL
model
CatBoost    0.636423
LightGBM    0.736358
Name: recall, dtype: float64

BEST ELIGIBLE RESULTS
   model  threshold  precision   recall       f1  true_positives  false_positives  false_negatives  true_negatives  missed_fraud_cost  false_alarm_cost  total_estimated_cost  cost_per_validation_transaction
LightGBM       0.05   0.251318 0.736358 0.374739            2240             6673              802           78866            80200.0           33365.0              113565.0                         1.282047


## Champion Model Selection

In [96]:
# CHAMPION MODEL SELECTION

# Select the lowest-cost model and threshold
# Select the lowest-cost candidate that meets the recall requirement
eligible_results = cost_results[
    cost_results["recall"] >= 0.70
]

assert not eligible_results.empty, (
    "No model meets the minimum recall requirement."
)

champion_row = eligible_results.loc[
    eligible_results["total_estimated_cost"].idxmin()
]

champion_model_name = champion_row["model"]
champion_threshold = float(champion_row["threshold"])

print("===== SELECTED CHAMPION =====")
print(f"Model: {champion_model_name}")
print(f"Threshold: {champion_threshold:.2f}")
print(f"Precision: {champion_row['precision']:.4f}")
print(f"Recall: {champion_row['recall']:.4f}")
print(f"F1: {champion_row['f1']:.4f}")
print(f"Fraud detected: {champion_row['true_positives']}")
print(f"Fraud missed: {champion_row['false_negatives']}")
print(f"False alarms: {champion_row['false_positives']}")
print(
    f"Estimated cost: "
    f"${champion_row['total_estimated_cost']:,.2f}"
)

# Save champion selection
import json
from pathlib import Path

champion_info = {
    "model": champion_model_name,
    "threshold": champion_threshold,
    "precision": float(champion_row["precision"]),
    "recall": float(champion_row["recall"]),
    "f1": float(champion_row["f1"]),
    "true_positives": int(champion_row["true_positives"]),
    "false_positives": int(champion_row["false_positives"]),
    "false_negatives": int(champion_row["false_negatives"]),
    "estimated_total_cost": float(
        champion_row["total_estimated_cost"]
    ),
    "cost_per_missed_fraud": COST_PER_MISSED_FRAUD,
    "cost_per_false_alarm": COST_PER_FALSE_ALARM,
    "cost_assumptions": "Illustrative"
}

champion_path = Path("artifacts/champion_model_selection.json")
champion_path.parent.mkdir(parents=True, exist_ok=True)

with open(champion_path, "w") as file:
    json.dump(champion_info, file, indent=4)

assert champion_path.exists()
assert champion_model_name == "LightGBM"

print(f"\nPASS: Champion selection saved to {champion_path}")

===== SELECTED CHAMPION =====
Model: LightGBM
Threshold: 0.05
Precision: 0.2513
Recall: 0.7364
F1: 0.3747
Fraud detected: 2240
Fraud missed: 802
False alarms: 6673
Estimated cost: $113,565.00

PASS: Champion selection saved to artifacts\champion_model_selection.json


## Connect to the shared MLflow database

In [97]:
import mlflow
import mlflow.lightgbm

# Use the same MLflow database as Credit Default Prediction
mlflow.set_tracking_uri(
    "sqlite:///C:/mlflow_new/mlflow.db"
)

# Create or select the FraudIntel experiment
mlflow.set_experiment("FraudIntel Fraud Detection")

print("===== MLFLOW CONNECTION =====")
print("Tracking URI:", mlflow.get_tracking_uri())
print(
    "Experiment:",
    mlflow.get_experiment_by_name(
        "FraudIntel Fraud Detection"
    ).name
)

2026/11/14 07:24:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/14 07:24:08 INFO mlflow.store.db.utils: Updating database tables
2026/11/14 07:24:10 INFO mlflow.tracking.fluent: Experiment with name 'FraudIntel Fraud Detection' does not exist. Creating a new experiment.


===== MLFLOW CONNECTION =====
Tracking URI: sqlite:///C:/mlflow_new/mlflow.db
Experiment: FraudIntel Fraud Detection


## Log the trained LightGBM model

In [98]:
import mlflow
import mlflow.lightgbm
from mlflow.models import infer_signature
from pathlib import Path

# Confirm the selected champion
assert champion_model_name == "LightGBM"
assert champion_row["recall"] >= 0.70

# Prepare model signature and example
input_example = X_val.head(5)
prediction_example = model_lgb.predict_proba(input_example)

signature = infer_signature(
    input_example,
    prediction_example
)

# Start the MLflow run
with mlflow.start_run(
    run_name="FraudIntel_LightGBM_Final_Champion"
) as run:

    # Log model parameters
    mlflow.log_params({
        key: str(value)
        for key, value in model_lgb.get_params().items()
    })

    # Log champion selection and business assumptions
    mlflow.log_param("selected_threshold", champion_threshold)
    mlflow.log_param("minimum_recall", 0.70)
    mlflow.log_param("cost_per_missed_fraud", COST_PER_MISSED_FRAUD)
    mlflow.log_param("cost_per_false_alarm", COST_PER_FALSE_ALARM)

    # Log evaluation metrics
    metrics = {
        "precision": float(champion_row["precision"]),
        "recall": float(champion_row["recall"]),
        "f1": float(champion_row["f1"]),
        "true_positives": int(champion_row["true_positives"]),
        "false_positives": int(champion_row["false_positives"]),
        "false_negatives": int(champion_row["false_negatives"]),
        "total_estimated_cost": float(
            champion_row["total_estimated_cost"]
        )
    }

    mlflow.log_metrics(metrics)

    # Log trained LightGBM model
    mlflow.lightgbm.log_model(
        model_lgb,
        artifact_path="lightgbm_model",
        signature=signature,
        input_example=input_example
    )

    # Log supporting reports
    mlflow.log_artifact(str(champion_path))
    mlflow.log_artifact(
        "reports/fraud_business_cost_analysis.csv"
    )
    mlflow.log_artifact(
        "reports/best_business_cost_by_model.csv"
    )

    final_run_id = run.info.run_id

print("===== MLFLOW LOGGING SUCCESSFUL =====")
print("Model:", champion_model_name)
print("Run ID:", final_run_id)
print("Threshold:", champion_threshold)
print(f"Recall: {metrics['recall']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"F1: {metrics['f1']:.4f}")
print(
    f"Estimated cost: "
    f"${metrics['total_estimated_cost']:,.2f}"
)
print("Tracking URI:", mlflow.get_tracking_uri())

C:\Users\NMP\PycharmProjects\FraudIntel\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/14 07:26:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/14 07:26:44 WARNING mlflow.utils.requirements_utils: Failed to run predict on input_example, dependencies int

===== MLFLOW LOGGING SUCCESSFUL =====
Model: LightGBM
Run ID: b7f9f77fa9e843e1a09298601648c929
Threshold: 0.05
Recall: 0.7364
Precision: 0.2513
F1: 0.3747
Estimated cost: $113,565.00
Tracking URI: sqlite:///C:/mlflow_new/mlflow.db


## Register the model

In [99]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:///C:/mlflow_new/mlflow.db"
)

model_name = "FraudIntel_LightGBM"

model_uri = (
    f"runs:/{final_run_id}/lightgbm_model"
)

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("===== MODEL REGISTRATION =====")
print("Model:", registered_model.name)
print("Version:", registered_model.version)
print("Run ID:", registered_model.run_id)

Successfully registered model 'FraudIntel_LightGBM'.
2026/11/14 07:28:25 WARNING mlflow.tracking._model_registry.fluent: Run with id b7f9f77fa9e843e1a09298601648c929 has no artifacts at artifact path 'lightgbm_model', registering model based on models:/m-5fd21a2b88ae42f7840abdc3498f9c82 instead


===== MODEL REGISTRATION =====
Model: FraudIntel_LightGBM
Version: 1
Run ID: b7f9f77fa9e843e1a09298601648c929


Created version '1' of model 'FraudIntel_LightGBM'.


## Verify the registered model

In [100]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
    "name='FraudIntel_LightGBM'"
)

for version in model_versions:
    print("Model:", version.name)
    print("Version:", version.version)
    print("Status:", version.status)
    print("Run ID:", version.run_id)
    print("Source:", version.source)

Model: FraudIntel_LightGBM
Version: 1
Status: READY
Run ID: b7f9f77fa9e843e1a09298601648c929
Source: models:/m-5fd21a2b88ae42f7840abdc3498f9c82


## Original in-memory model probabilities

In [105]:
# Original in-memory model probabilities
original_probabilities = model_lgb.predict_proba(X_val)[:, 1]

# Registered model probabilities
if hasattr(loaded_model, "predict_proba"):
    registered_probabilities = loaded_model.predict_proba(X_val)[:, 1]
else:
    registered_probabilities = loaded_model.predict(X_val)

print("===== MODEL CONSISTENCY CHECK =====")

print(
    "Original model predictions:",
    len(original_probabilities)
)

print(
    "Registered model predictions:",
    len(registered_probabilities)
)

print(
    "\nOriginal probabilities:",
    original_probabilities[:10]
)

print(
    "\nRegistered outputs:",
    registered_probabilities[:10]
)

print(
    "\nOriginal fraud predictions at 0.05:",
    int((original_probabilities >= champion_threshold).sum())
)

print(
    "Registered fraud predictions at 0.05:",
    int((registered_probabilities >= champion_threshold).sum())
)


===== MODEL CONSISTENCY CHECK =====
Original model predictions: 88581
Registered model predictions: 88581

Original probabilities: [0.00585582 0.80693298 0.0949062  0.00665985 0.00380772 0.00706431
 0.05523059 0.03148151 0.04780365 0.02648803]

Registered outputs: [0.00585582 0.80693298 0.0949062  0.00665985 0.00380772 0.00706431
 0.05523059 0.03148151 0.04780365 0.02648803]

Original fraud predictions at 0.05: 8913
Registered fraud predictions at 0.05: 8913


## Create and log the inference configuration

In [106]:
from pathlib import Path

# Inference configuration
inference_config = {
    "model_name": "FraudIntel_LightGBM",
    "model_version": 1,
    "model_type": "LightGBM",
    "threshold": champion_threshold,
    "prediction_method": "predict_proba",
    "positive_class": 1,
    "minimum_recall_requirement": 0.70,
    "cost_per_missed_fraud": COST_PER_MISSED_FRAUD,
    "cost_per_false_alarm": COST_PER_FALSE_ALARM,
    "cost_assumptions": "Illustrative"
}

# Save locally
config_path = Path("artifacts/inference_config.json")
config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as file:
    json.dump(inference_config, file, indent=4)

# Log configuration to the existing MLflow run
mlflow.set_tracking_uri(
    "sqlite:///C:/mlflow_new/mlflow.db"
)

with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(str(config_path))

print("===== INFERENCE CONFIGURATION =====")
print(json.dumps(inference_config, indent=4))
print("\nPASS: Configuration saved locally and logged to MLflow.")

===== INFERENCE CONFIGURATION =====
{
    "model_name": "FraudIntel_LightGBM",
    "model_version": 1,
    "model_type": "LightGBM",
    "threshold": 0.05,
    "prediction_method": "predict_proba",
    "positive_class": 1,
    "minimum_recall_requirement": 0.7,
    "cost_per_missed_fraud": 100.0,
    "cost_per_false_alarm": 5.0,
    "cost_assumptions": "Illustrative"
}

PASS: Configuration saved locally and logged to MLflow.


## Build the fraud prediction function

In [107]:
# Load the registered model
mlflow.set_tracking_uri(
    "sqlite:///C:/mlflow_new/mlflow.db"
)

registered_model = mlflow.lightgbm.load_model(
    "models:/FraudIntel_LightGBM/1"
)

# Load the saved inference configuration
with open("artifacts/inference_config.json", "r") as file:
    inference_config = json.load(file)

FRAUD_THRESHOLD = inference_config["threshold"]


# Reusable fraud prediction function
def predict_fraud(data):

    # Generate fraud probabilities
    probabilities = registered_model.predict_proba(data)[:, 1]

    # Apply the selected threshold
    predictions = (
        probabilities >= FRAUD_THRESHOLD
    ).astype(int)

    # Return structured results
    results = pd.DataFrame(
        {
            "fraud_probability": probabilities,
            "fraud_prediction": predictions
        },
        index=data.index
    )

    return results


# Test the function
test_results = predict_fraud(X_val.head(10))

print("===== FRAUD INFERENCE TEST =====")
print(test_results)

print("\nFraud cases detected:", test_results["fraud_prediction"].sum())
print("Threshold:", FRAUD_THRESHOLD)

assert len(test_results) == 10
assert test_results["fraud_probability"].between(0, 1).all()

print("\nPASS: Reusable fraud prediction function working.")

===== FRAUD INFERENCE TEST =====
   fraud_probability  fraud_prediction
0           0.005856                 0
1           0.806933                 1
2           0.094906                 1
3           0.006660                 0
4           0.003808                 0
5           0.007064                 0
6           0.055231                 1
7           0.031482                 0
8           0.047804                 0
9           0.026488                 0

Fraud cases detected: 3
Threshold: 0.05

PASS: Reusable fraud prediction function working.


In [111]:
categorical_columns = X_train.select_dtypes(
    include=["category"]
).columns.tolist()

print("TOTAL FEATURES:", len(X_train.columns))
print("CATEGORICAL FEATURES:", len(categorical_columns))
print("\nCATEGORICAL COLUMN NAMES:")
print(categorical_columns)

TOTAL FEATURES: 432
CATEGORICAL FEATURES: 31

CATEGORICAL COLUMN NAMES:
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


## inference schema

In [112]:

# Get the trained categorical column names
categorical_columns = X_train.select_dtypes(
    include=["category"]
).columns.tolist()

# Get the LightGBM Booster
booster = model_lgb.booster_

# Build category metadata
categorical_schema = {}

for column_name, categories in zip(
    categorical_columns,
    booster.pandas_categorical
):
    categorical_schema[column_name] = [
        str(category) for category in categories
    ]

# Build complete inference schema
inference_schema = {
    "feature_count": len(booster.feature_name()),
    "feature_names": booster.feature_name(),
    "categorical_columns": categorical_columns,
    "categorical_categories": categorical_schema
}

# Save artifact
schema_path = Path("artifacts/inference_schema.json")
schema_path.parent.mkdir(parents=True, exist_ok=True)

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(inference_schema, f, indent=4)

# Verify
assert inference_schema["feature_count"] == 432
assert len(inference_schema["categorical_columns"]) == 31
assert len(inference_schema["categorical_categories"]) == 31

print("PASS: Inference schema saved")
print(f"Path: {schema_path}")
print(f"Features: {inference_schema['feature_count']}")
print(f"Categorical columns: {len(inference_schema['categorical_columns'])}")

PASS: Inference schema saved
Path: artifacts\inference_schema.json
Features: 432
Categorical columns: 31


In [113]:
# Export one real validation transaction for API testing
sample_transaction = X_val.iloc[[0]].copy()

sample_transaction.to_csv(
    "artifacts/sample_transaction.csv",
    index=False
)

print("Sample transaction exported successfully")
print("Shape:", sample_transaction.shape)

Sample transaction exported successfully
Shape: (1, 432)
